# HMM Regime App

An ipywidgets app, built the same way as the Crypto App, that:

1. **loads prices** from Yahoo Finance with the same universe / search / currency inputs as the Multi Asset App (ETF, stocks, crypto such as `BTC-USD`), or from a CSV / Excel file,
2. **fits a Gaussian Hidden Markov Model** with user settings (features, number of states, covariance type, training window, walk-forward refits...),
3. **backtests strategies mapped to each regime**: every regime gets its own optimizer, exposure, asset constraints and risk limits (volatility, tracking error, information ratio, beta).

The notebook is built step by step: each step defines functions that the next step uses, and Step 4 assembles them into the app.

**No look-ahead.** By default the regime on day *t* comes from the *filtered* probability P(state_t | data up to t), computed with the forward algorithm. Weights decided at the close of *t* earn the returns of *t+1*. The smoothed (forward-backward) view is available for analysis only.

## Step 0 - Imports

In [ ]:
import io
import os
import copy
import json
import logging
import datetime
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count

import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from scipy.optimize import linear_sum_assignment

import ipywidgets as widgets
from ipydatagrid import DataGrid
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

from src.RiskMetrics import (RiskAnalysis, build_constraint, vol_constraint,
                             tracking_error_constraint, information_ratio_constraint,
                             beta_constraint, compute_per_asset_betas)
from src.Metrics import display_scrollable_df, get_asset_returns, get_asset_risk, rebalanced_metrics
# all yfinance access -- imported last so nothing above can shadow these names (as in the Multi Asset App)
from src.Yahoo_API import (QUOTE_TYPE_CLASS, search_yahoo, fetch_ticker_details, has_price_history,
                           get_close, get_currencies, get_fx_rates, to_base_currency)

# hmmlearn logs "Model is not converging" at every n_init restart; the fit
# keeps the best restart anyway, so the noise is silenced.
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

## Step 1 - Price loading

Same inputs as the Multi Asset App: a predefined universe (ticker + asset class) that you tick / untick, a Yahoo Finance search bar to add any ticker, optional ticker details, and prices converted to a base currency. Prices come from `src.Yahoo_API.get_close` (adjusted closes). A CSV / Excel file can be loaded instead.

Everything returns a clean `DataFrame` of prices (dates × assets), so the rest of the app does not care whether the prices are ETFs, stocks or crypto (e.g. `BTC-USD`).

In [ ]:
# Predefined universe -- same tickers and asset classes as the Multi Asset App
UNIVERSE = {
    'Commodities': ['ISLN.L', 'GOLD.PA', 'BZ=F', 'DBC'],
    'Bonds': ['IEF', 'IEI'],
    'Credit': ['DHY', 'LQD'],
    'T-Bills': ['BIL'],
    'Equity': ['^GSPC', 'IDU', 'XLK', 'SOXX', 'XLP', 'XLY', 'IYJ', 'MXI', 'IYE',
               'PICK', 'EXV5.DE', 'IUHC.L', 'IYG', 'XSW'],
}
# asset classes held by the strategies but left out of the HMM feature baskets
NO_FEATURE_BASKET = {'T-Bills'}
BASE_CURRENCY = 'USD'
CONVERT_TO_BASE = True
BASE_CURRENCY_CHOICES = ('USD', 'EUR', 'GBP', 'CHF', 'JPY', 'HUF', 'CAD', 'AUD', 'SEK', 'NOK', 'DKK', 'PLN', 'CZK')


def clean_prices(prices, max_missing=0.2, ffill_limit=5):
    """Sorted, de-duplicated, numeric prices.

    Short gaps (holidays) are forward-filled up to `ffill_limit` days. An asset
    is dropped when more than `max_missing` of its history *after its first
    price* is missing -- an asset listed late is kept, it just enters the
    backtest when it has enough history.
    """
    if prices is None or prices.empty:
        return pd.DataFrame()
    prices = prices.copy()
    prices.index = pd.to_datetime(prices.index)
    if prices.index.tz is not None:
        prices.index = prices.index.tz_localize(None)
    prices = prices[~prices.index.duplicated(keep='last')].sort_index()
    prices = prices.apply(pd.to_numeric, errors='coerce')
    prices = prices.where(prices > 0)
    prices = prices.dropna(axis=1, how='all').dropna(axis=0, how='all')
    prices = prices.ffill(limit=ffill_limit)
    keep = [c for c in prices.columns
            if prices[c].loc[prices[c].first_valid_index():].isna().mean() <= max_missing]
    return prices[keep]


def read_price_file(file_path=None, upload=None):
    """Prices from a CSV / Excel file: first column = dates, one column per asset.

    `upload` is the value of a widgets.FileUpload (ipywidgets 8: a tuple of
    dicts with 'name' and 'content'); it takes precedence over `file_path`.
    """
    if upload:
        file = upload[0]
        name, source = file['name'], io.BytesIO(bytes(file['content']))
    elif file_path:
        name, source = file_path, file_path
    else:
        raise ValueError("Provide a file path or upload a file.")

    if name.lower().endswith(('.xlsx', '.xls')):
        data = pd.read_excel(source, index_col=0)
    else:
        data = pd.read_csv(source, index_col=0)
    data.index = pd.to_datetime(data.index, errors='coerce')
    return data[data.index.notna()]


def load_yahoo_prices(tickers, start, end=None, base_currency=BASE_CURRENCY, convert=CONVERT_TO_BASE, on_progress=None):
    """Adjusted closes from Yahoo Finance, optionally converted to `base_currency`.

    Returns (prices, messages). `on_progress(ticker, ok)` is called as each
    download finishes (drives the progress bar)."""
    end = end or datetime.date.today()
    messages = []
    prices = get_close(tickers, start=start, end=end, max_workers=8, on_progress=on_progress)
    if prices.attrs.get('failed'):
        messages.append(f"⚠️ No data returned for: {', '.join(prices.attrs['failed'])}")
    if prices.empty:
        return prices, messages

    currencies = get_currencies(prices.columns).fillna(base_currency)
    if convert:
        fx = get_fx_rates(currencies.values, base_currency, start=start, end=end)
        if fx.attrs.get('missing'):
            messages.append(f"⚠️ No FX rate for {', '.join(fx.attrs['missing'])} -> {base_currency}; left unconverted")
        prices = to_base_currency(prices, currencies, fx)
        foreign = currencies[currencies != base_currency]
        if len(foreign):
            messages.append(f"💱 Converted to {base_currency}: " + ", ".join(f"{t} ({c})" for t, c in foreign.items()))
    else:
        mixed = sorted(set(currencies.dropna()))
        if len(mixed) > 1:
            messages.append(f"⚠️ Prices left in local currencies ({', '.join(mixed)}) -- returns and risk mix currencies.")
    return clean_prices(prices), messages

## Step 2 - HMM engine

**2.a Features.** The HMM sees a few causal market features.

The features are computed on one **equal-weight basket per asset class** of the universe (Equity, Bonds, Credit, Commodities...), the same idea as the asset-class factors of `HMM.ipynb`:

| Feature | Definition |
|---|---|
| Returns | rolling mean of each basket's log returns over the smoothing window, annualized |
| Volatility | log of each basket's rolling annualized volatility |
| Market Factor (PCA) | 1st principal component of the baskets' z-scored returns, loadings re-estimated at each date on the past 252 observations (rolling, causal), smoothed |
| Average Correlation | rolling average pairwise correlation between the baskets |
| Drawdown | drawdown of the equal-weight basket of all asset classes from its 1-year high |
| Index Return | rolling mean of the index's log returns over the vol window, annualized |
| Index Volatility | log of the index's rolling annualized volatility |
| Index Drawdown | drawdown of the index from its 1-year high |

The three **Index** features use a single index (default `^GSPC`) loaded from its own start date, so the model can train on a longer history than the universe (e.g. from 2007, with the 2008 crisis in the training data). T-Bills are held by the strategies but get no feature basket.

**2.b Fit.** Features are standardized with training-window statistics, then a `GaussianHMM` is fitted `n_init` times from different seeds and the best log-likelihood is kept.

**2.c State ordering.** HMM states have arbitrary numbers, so they are relabelled by the volatility (or mean return) of the feature basket in each state: `R0` = lowest volatility. With **Keep labels across refits**, only the first fit is ordered that way: every later fit's states are matched to it (closest state means, Hungarian assignment), so a label means the same market condition in 2020 and in 2026 instead of being re-ranked at each refit.

**2.d Filtering.** `forward_filter` runs the forward algorithm to get P(state_t | x_1..x_t). With walk-forward refits, each model only produces probabilities for the period after its fit date.

**2.e Training window.** *Growing* fits each model on everything from Training Start to its fit date (expanding window). *Sliding* fits each model on the last *N* years only (never before Training Start), so the model adapts faster but can lose rare regimes.

**2.f Signal confirmation.** You can require a minimum probability and *k* consecutive observations before switching regime, which reduces whipsaws.

In [ ]:
# computed on one index (e.g. ^GSPC) loaded from its own start date, not on the baskets
INDEX_FEATURES = ['Index Return', 'Index Volatility', 'Index Drawdown']
FEATURE_TYPES = ['Returns', 'Volatility', 'Market Factor (PCA)', 'Average Correlation', 'Drawdown'] + INDEX_FEATURES
COVARIANCE_TYPES = ['full', 'diag', 'spherical', 'tied']
REFIT_OPTIONS = {'Never (single fit)': None, 'Monthly': 'ME', 'Quarterly': 'QE', 'Yearly': 'YE'}
WINDOW_OPTIONS = ['Growing (expanding)', 'Sliding']
ORDER_OPTIONS = ['Volatility (low → high)', 'Return (high → low)']
INFERENCE_OPTIONS = ['Filtered (causal)', 'Smoothed (look-ahead, analysis only)']


def periods_per_year(index):
    """365 when the index trades on weekends (crypto), else 252."""
    index = pd.DatetimeIndex(index)
    if len(index) < 10:
        return 252
    return 365 if (index.dayofweek >= 5).mean() > 0.1 else 252


def asset_groups(columns, universe_table=None):
    """{asset class: [tickers]} for the loaded assets, from the universe table
    ('Asset Class' column). Tickers it doesn't know go to 'Other'."""
    classes = universe_table['Asset Class'] if universe_table is not None else pd.Series(dtype=object)
    classes = classes[~classes.index.duplicated()].reindex(list(columns)).fillna('Other')
    return {g: list(classes.index[classes == g]) for g in dict.fromkeys(classes)}


def group_returns(prices, groups, frequency='Daily'):
    """Log return of the equal-weight basket of each group (asset class).

    An asset joins its basket once it has prices; the history starts when
    every group has at least one asset."""
    px_ = prices.resample('W-FRI').last() if frequency == 'Weekly' else prices
    rets = np.log(px_).diff()
    baskets = pd.DataFrame({g: rets[list(t)].mean(axis=1) for g, t in groups.items() if len(t)})
    return baskets.iloc[1:].dropna()


def reference_returns(prices, groups, frequency='Daily'):
    """Log return of the equal-weight basket of all groups -- used to order
    and describe the states."""
    return group_returns(prices, groups, frequency).mean(axis=1).rename('Reference')


def index_returns(index_prices, frequency='Daily'):
    """Log return of the index behind the Index features -- used instead of
    the baskets to order and describe the states when those features are on."""
    px_ = index_prices.dropna()
    px_ = px_.resample('W-FRI').last() if frequency == 'Weekly' else px_
    return np.log(px_).diff().iloc[1:].rename('Reference')


def build_features(prices, groups, feature_types, vol_window=21, smooth_window=5, frequency='Daily', fit_end=None,
                   index_prices=None, pca_window=252):
    """HMM features computed on the asset-class baskets of `groups`, and the
    Index features on `index_prices` (one index, e.g. ^GSPC).

    pca_window: observations the Market Factor loadings are re-estimated on at
    each date (rolling, causal). None = one fixed set of loadings fitted up to
    `fit_end`."""
    if not feature_types:
        raise ValueError("Select at least one feature type.")
    if not groups and set(feature_types) - set(INDEX_FEATURES):
        raise ValueError("No assets loaded.")

    rets = group_returns(prices, groups, frequency) if groups else pd.DataFrame()
    assets = list(rets.columns)
    ann = 52 if frequency == 'Weekly' else periods_per_year(rets.index)
    year = 52 if frequency == 'Weekly' else ann
    parts = []

    if 'Returns' in feature_types:
        f = rets.rolling(smooth_window).mean() * ann
        parts.append(f.add_suffix(' | Return'))

    if 'Volatility' in feature_types:
        # log-vol is much closer to Gaussian than vol itself
        f = np.log(rets.rolling(vol_window).std() * np.sqrt(ann))
        parts.append(f.add_suffix(' | Log Vol'))

    if 'Market Factor (PCA)' in feature_types:
        if len(assets) > 1 and pca_window:
            # loadings, mean and std re-estimated each date on the last pca_window observations
            x = rets.dropna()
            vals = x.values
            factor = np.full(len(x), np.nan)
            for t in range(pca_window - 1, len(x)):
                w = vals[t - pca_window + 1:t + 1]
                mu, sd = w.mean(axis=0), w.std(axis=0, ddof=1)
                loadings = np.linalg.eigh(np.corrcoef(w, rowvar=False))[1][:, -1]
                loadings = loadings * np.sign(loadings.sum() or 1)
                factor[t] = ((vals[t] - mu) / sd) @ loadings
            factor = pd.Series(factor, index=x.index).reindex(rets.index)
        elif len(assets) > 1:
            fit_rets = rets.loc[:fit_end].dropna() if fit_end is not None else rets.dropna()
            mu, sd = fit_rets.mean(), fit_rets.std()
            z = (rets - mu) / sd
            loadings = PCA(n_components=1).fit(((fit_rets - mu) / sd).values).components_[0]
            loadings = loadings * np.sign(loadings.sum() or 1)
            factor = z.fillna(0) @ loadings
        else:
            factor = rets.iloc[:, 0]
        parts.append((factor.rolling(smooth_window).mean() * ann).rename('Market Factor'))

    if 'Average Correlation' in feature_types and len(assets) > 1:
        n = len(assets)
        corr_sum = rets.rolling(vol_window).corr().groupby(level=0).sum(min_count=1).sum(axis=1, min_count=1)
        parts.append(((corr_sum - n) / (n * (n - 1))).rename('Average Correlation'))

    if 'Drawdown' in feature_types:
        basket = np.exp(rets.mean(axis=1).fillna(0).cumsum())
        parts.append((basket / basket.rolling(year, min_periods=vol_window).max() - 1).rename('Basket Drawdown'))

    index_types = [f for f in feature_types if f in INDEX_FEATURES]
    if index_types:
        if index_prices is None or index_prices.dropna().empty:
            raise ValueError("The Index features need the index prices.")
        px_ = index_prices.dropna()
        px_ = px_.resample('W-FRI').last() if frequency == 'Weekly' else px_
        r = np.log(px_).diff()
        ann_idx = 52 if frequency == 'Weekly' else periods_per_year(px_.index)
        if 'Index Return' in index_types:
            parts.append((r.rolling(vol_window).mean() * ann_idx).rename('Index Return'))
        if 'Index Volatility' in index_types:
            parts.append(np.log(r.rolling(vol_window).std() * np.sqrt(ann_idx)).rename('Index Log Vol'))
        if 'Index Drawdown' in index_types:
            parts.append((px_ / px_.rolling(ann_idx, min_periods=ann_idx // 2).max() - 1).rename('Index Drawdown'))

    if not parts:
        raise ValueError("The selected feature types need at least 2 asset classes.")
    return pd.concat(parts, axis=1).replace([np.inf, -np.inf], np.nan).dropna()


def fit_hmm(X, n_states, covariance_type='full', n_iter=500, tol=1e-4, n_init=5, random_state=42):
    """Best of `n_init` GaussianHMM fits on standardized X.

    Returns (model, scaler, log_likelihood)."""
    scaler = StandardScaler().fit(X)
    Z = scaler.transform(X)
    best, best_score = None, -np.inf
    for i in range(max(1, n_init)):
        model = GaussianHMM(n_components=n_states, covariance_type=covariance_type, n_iter=n_iter,
                            tol=tol, random_state=random_state + i, min_covar=1e-3)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                model.fit(Z)
                score = model.score(Z)
        except Exception:
            continue
        if np.isfinite(score) and score > best_score:
            best, best_score = model, score
    if best is None:
        raise RuntimeError("HMM failed to fit -- try fewer states, 'diag' covariance or fewer features.")
    return best, scaler, best_score


def state_order(model, Z, ref, order_by):
    """order[new_label] = raw HMM state. R0 = lowest vol (or highest return)."""
    raw = model.predict(Z)
    vols, means = [], []
    for k in range(model.n_components):
        r = ref[raw == k]
        vols.append(r.std() if len(r) > 1 else np.inf)
        means.append(r.mean() if len(r) else -np.inf)
    if order_by.startswith('Return'):
        return np.argsort(-np.nan_to_num(means, nan=-np.inf))
    return np.argsort(np.nan_to_num(vols, nan=np.inf))


def match_state_order(means, anchor_means, scale):
    """order[label] = raw state whose mean (in feature units) is closest to the
    label's mean in the first fit, so a label keeps its meaning across refits."""
    cost = np.linalg.norm((means[:, None, :] - anchor_means[None, :, :]) / scale, axis=2)
    rows, cols = linear_sum_assignment(cost)
    order = np.empty(len(cols), dtype=int)
    order[cols] = rows
    return order


def emission_loglik(model, Z):
    """log N(z_t | mean_k, cov_k), shape (T, n_states)."""
    try:
        # hmmlearn's own emission term -- exact for every covariance_type
        return model._compute_log_likelihood(Z)
    except AttributeError:
        pass
    # Fallback if the private method disappears. Note: hmmlearn 0.3.3's
    # covars_ is mis-shaped for 'spherical', hence the preference above.
    covars = model.covars_
    return np.column_stack([
        np.atleast_1d(multivariate_normal(model.means_[k], covars[k], allow_singular=True).logpdf(Z))
        for k in range(model.n_components)
    ])


def forward_filter(model, Z):
    """Filtered state probabilities P(s_t | z_1..z_t) via the forward algorithm.

    Unlike model.predict_proba (forward-backward), row t only uses data up
    to t, so it is safe to trade on."""
    log_b = emission_loglik(model, Z)
    log_a = np.log(np.maximum(model.transmat_, 1e-300))
    log_alpha = np.empty_like(log_b)
    a = np.log(np.maximum(model.startprob_, 1e-300)) + log_b[0]
    log_alpha[0] = a - logsumexp(a)
    for t in range(1, len(Z)):
        a = logsumexp(log_alpha[t - 1][:, None] + log_a, axis=0) + log_b[t]
        log_alpha[t] = a - logsumexp(a)
    return np.exp(log_alpha)


def confirm_regimes(probs, min_prob=0.0, confirm=1):
    """Most likely regime, switching only when the new regime has probability
    >= min_prob for `confirm` consecutive observations."""
    values = probs.values
    raw, pmax = values.argmax(axis=1), values.max(axis=1)
    out = np.empty(len(raw), dtype=int)
    current, candidate, streak = raw[0], raw[0], 0
    for t in range(len(raw)):
        proposal = raw[t] if pmax[t] >= min_prob else current
        if proposal == current:
            streak = 0
        else:
            streak = streak + 1 if proposal == candidate else 1
            candidate = proposal
            if streak >= confirm:
                current, streak = proposal, 0
        out[t] = current
    return pd.Series(out, index=probs.index, name='Regime')


def run_regime_model(features, ref_returns, train_start, train_end, n_states=3, covariance_type='full',
                     n_iter=500, tol=1e-4, n_init=5, random_state=42, refit='Never (single fit)',
                     order_by=ORDER_OPTIONS[0], inference=INFERENCE_OPTIONS[0], min_prob=0.0, confirm=1,
                     window=WINDOW_OPTIONS[0], window_years=5.0, match_states=False):
    feats = features.loc[pd.Timestamp(train_start):]
    train_end = pd.Timestamp(train_end)
    sliding = window == 'Sliding'
    if sliding and not window_years > 0:
        raise ValueError("The sliding window length must be positive.")
    min_obs = max(60, 15 * n_states)

    def fit_mask(fit_date):
        """Observations a model fitted at fit_date trains on."""
        mask = feats.index <= fit_date
        if sliding:
            mask &= feats.index > fit_date - pd.DateOffset(days=int(round(365.25 * window_years)))
        return mask

    n_train = int(fit_mask(train_end).sum())
    if n_train < min_obs:
        raise ValueError(f"Only {n_train} training observations -- extend the training window"
                         + (" or the sliding length." if sliding else "."))

    values = feats.values
    ref = ref_returns.reindex(feats.index).fillna(0).values
    smoothed = inference.startswith('Smoothed')

    fit_dates = [train_end]
    rule = REFIT_OPTIONS.get(refit)
    if rule and not smoothed:
        fit_dates += [d for d in pd.date_range(train_end, feats.index[-1], freq=rule) if d > train_end]

    parts, fits, anchor = [], [], None
    for i, fit_date in enumerate(fit_dates):
        in_fit = fit_mask(fit_date)
        if in_fit.sum() < min_obs:
            raise ValueError(f"Only {int(in_fit.sum())} observations in the window fitted on {fit_date.date()} "
                             "-- lengthen the sliding window.")
        model, scaler, loglik = fit_hmm(values[in_fit], n_states, covariance_type, n_iter, tol, n_init, random_state)
        Z = scaler.transform(values)
        means = scaler.inverse_transform(model.means_)   # state means in feature units
        if match_states and anchor is not None:
            order = match_state_order(means, *anchor)
        else:
            order = state_order(model, Z[in_fit], ref[in_fit], order_by)
        if anchor is None:
            # the first fit defines what each label means; matched refits keep it
            anchor = (means[order], values[in_fit].std(axis=0))

        seg_end = fit_dates[i + 1] if i + 1 < len(fit_dates) else feats.index[-1]
        upto = feats.index <= seg_end
        probs = model.predict_proba(Z[upto]) if smoothed else forward_filter(model, Z[upto])
        probs = probs[:, order]
        keep = np.ones(upto.sum(), dtype=bool) if i == 0 else feats.index[upto] > fit_date
        parts.append(pd.DataFrame(probs[keep], index=feats.index[upto][keep]))
        fits.append({'Fit date': fit_date, 'Window start': feats.index[in_fit][0],
                     'Observations': int(in_fit.sum()), 'Log-Likelihood': loglik,
                     'model': model, 'scaler': scaler, 'order': order})

    probs = pd.concat(parts)
    probs.columns = list(range(n_states))
    last = fits[-1]
    transmat = pd.DataFrame(last['model'].transmat_[np.ix_(last['order'], last['order'])],
                            index=range(n_states), columns=range(n_states))
    return {
        'probs': probs,
        'regimes': confirm_regimes(probs, min_prob, confirm),
        'fits': fits,
        'transmat': transmat,
        'features': feats,
        'train_end': train_end,
    }


def select_n_states(features, train_start, train_end, max_states=6, covariance_type='full',
                    n_iter=500, tol=1e-4, n_init=3, random_state=42):
    """Information criteria for 2..max_states on the training window (lower BIC = better)."""
    X = features.loc[pd.Timestamp(train_start):pd.Timestamp(train_end)].values
    rows = []
    for k in range(2, max_states + 1):
        try:
            model, scaler, loglik = fit_hmm(X, k, covariance_type, n_iter, tol, n_init, random_state)
        except RuntimeError:
            continue
        Z = scaler.transform(X)
        rows.append({'States': k, 'Log-Likelihood': loglik, 'AIC': model.aic(Z), 'BIC': model.bic(Z)})
    return pd.DataFrame(rows).set_index('States').round(1)


def regime_summary(regimes, transmat, labels):
    runs = (regimes != regimes.shift()).cumsum()
    episodes = regimes.groupby(runs).agg(['first', 'size'])
    summary = pd.DataFrame({
        '% of Time': regimes.value_counts(normalize=True),
        'Episodes': episodes.groupby('first').size(),
        'Avg Duration (obs)': episodes.groupby('first')['size'].mean(),
    }).reindex(transmat.index)
    summary['Expected Duration (model)'] = 1 / (1 - np.diag(transmat.values)).clip(1e-9)
    summary.index = [labels[k] for k in summary.index]
    return summary.round(3)


def regime_asset_statistics(prices, regimes_daily, ann):
    """Annualized return / vol / Sharpe of every asset, conditional on the regime of the day."""
    rets = prices.pct_change(fill_method=None)
    grouped = rets.groupby(regimes_daily)
    mean, vol = grouped.mean() * ann, grouped.std() * np.sqrt(ann)
    return mean, vol, mean / vol


def empirical_transitions(regimes):
    return pd.crosstab(regimes.shift().dropna().astype(int), regimes.iloc[1:], normalize='index')

## Step 3 - Regime-mapped backtest engine

Each regime is mapped to:

* a **strategy**: one of the `RiskAnalysis.optimize` objectives (Minimum Variance, Risk Parity, Sharpe Ratio, Maximum Diversification, Eigen), or Equal Weight, Inverse Volatility, Custom Weights, Cash,
* an **exposure** (1 = fully invested, 0.5 = half in cash),
* its own **asset constraints** (`Asset / All`, `=, ≥, ≤`, limit) and **risk limits** (volatility, tracking error, information ratio, beta), built with the same `build_constraint` / `vol_constraint` / `tracking_error_constraint` / `information_ratio_constraint` / `beta_constraint` as the Crypto App. An asset's own row overrides `All` for that asset (`All ≤ 0.35` + `BIL ≤ 0.6`). An asset capped at 0 is left out of that regime whatever the strategy, Equal Weight included (`BIL ≤ 0` keeps T-bills out). Tracking error, information ratio and beta are measured against the *TE / IR / Beta benchmark*: equal weight of the eligible assets, or 100% in one asset (e.g. `^GSPC`); betas are estimated on each decision's lookback window.

**Rebalancing.** Weights are recomputed on the calendar dates (weekly / monthly / quarterly / yearly) and, optionally, whenever the regime changes. Each decision uses the regime known at that close and the last `lookback` days of returns. Optionally, only the past days that were in the *same regime* are used, which gives a regime-conditional covariance.

**Simulation.** Weights drift with prices between rebalances. Cash earns 0. Costs = turnover × bps.

In [ ]:
STRATEGIES = {
    'Minimum Variance': 'minimum_variance',
    'Risk Parity': 'risk_parity',
    'Sharpe Ratio': 'sharpe_ratio',
    'Maximum Diversification': 'maximum_diversification',
    'Eigen Strategy': 'eigenportfolio',
    'Minimum Tracking Error': 'tracking_error',        # vs the TE / IR benchmark
    'Maximum Information Ratio': 'information_ratio',  # vs the TE / IR benchmark
    'Equal Weight': 'equal_weight',
    'Inverse Volatility': 'inverse_vol',
    'Custom Weights': 'custom',
    'Cash': 'cash',
}
OPTIMIZED = {'minimum_variance', 'risk_parity', 'sharpe_ratio', 'maximum_diversification', 'eigenportfolio',
             'tracking_error', 'information_ratio'}
RELATIVE = {'tracking_error', 'information_ratio'}   # need the benchmark weights
REBALANCING = {'Weekly': 'W-FRI', 'Monthly': 'ME', 'Quarterly': 'QE', 'Yearly': 'YE'}
RISK_TYPES = ['Volatility', 'Tracking Error', 'Information Ratio', 'Beta']


def period_end_dates(index, frequency):
    """Last available date of each calendar period."""
    s = pd.Series(index, index=index)
    return pd.DatetimeIndex(s.groupby(pd.Grouper(freq=REBALANCING[frequency])).last().dropna().values)


def benchmark_weights(columns, te_benchmark='Equal Weight'):
    """TE / IR benchmark over the eligible assets: equal weight, or 100% in
    one asset (equal weight when that asset isn't eligible at this date)."""
    if te_benchmark in columns:
        return pd.Series(np.where(columns == te_benchmark, 1.0, 0.0), index=columns)
    return pd.Series(1 / len(columns), index=columns)


def expand_all_rows(asset_constraints, columns):
    """'All' rows -> one row per asset, except the assets with their own row of the
    same sign: All ≤ 0.35 + BIL ≤ 0.6 caps BIL at 60% and every other asset at 35%."""
    own = {(asset, sign) for asset, sign, _ in asset_constraints if asset != 'All'}
    rows = []
    for asset, sign, limit in asset_constraints:
        if asset == 'All':
            rows += [[c, sign, limit] for c in columns if (c, sign) not in own and (c, '=') not in own]
        else:
            rows.append([asset, sign, limit])
    return rows


def compute_target_weights(window, strategy, asset_constraints=None, risk_limits=None, custom=None, bench=None):
    """Fully-invested target weights (before exposure scaling) for one decision.

    window: returns of the eligible assets. asset_constraints: rows
    [Asset, Sign, Limit]. risk_limits: rows [Risk, Sign, Limit] with Risk in
    RISK_TYPES (annualized; TE in decimals, IR and beta as ratios). bench: benchmark
    weights for TE / IR / beta (Series over window's columns). custom: Series of weights.
    An asset capped at 0 (sign ≤ or = with limit 0) is left out of the window,
    so every strategy leaves it out (Equal Weight and Custom Weights included).
    Returns (weights Series, note)."""
    # benchmark returns for a Beta limit, taken before zero-capped assets are left out
    # (a beta to ^GSPC stays a beta to ^GSPC in a regime that can't hold ^GSPC)
    bench_returns = None
    if any(row[0] == 'Beta' for row in (risk_limits or [])):
        b = bench if bench is not None else benchmark_weights(window.columns)
        bench_returns = window @ b.reindex(window.columns).fillna(0.0)
    zero = {a for a, sign, limit in (asset_constraints or [])
            if a != 'All' and sign in ('≤', '=') and float(limit) <= 0}
    if zero:
        window = window.loc[:, [c for c in window.columns if c not in zero]]
        if bench is not None:
            bench = bench.reindex(window.columns).fillna(0.0)
            bench = bench / bench.sum() if bench.sum() > 0 else None
    key = STRATEGIES[strategy]
    cols = window.columns
    n = len(cols)
    if key == 'cash' or n == 0:
        return pd.Series(0.0, index=cols), ''
    if key == 'custom':
        return (custom if custom is not None else pd.Series(dtype=float)).reindex(cols).fillna(0.0), ''
    if key == 'equal_weight' or n == 1:
        return pd.Series(1 / n, index=cols), ''
    if key == 'inverse_vol':
        inv = 1 / window.std().replace(0, np.nan)
        return (inv / inv.sum()).fillna(0.0), ''

    bench = bench if bench is not None else benchmark_weights(cols)
    cons = []
    if asset_constraints:
        cons += build_constraint(window, np.array(expand_all_rows(asset_constraints, cols), dtype=object))
    for risk, sign, limit in (risk_limits or []):
        if risk == 'Volatility':
            cons += vol_constraint(window.cov().values, sign, float(limit))
        elif risk == 'Tracking Error':
            cons += tracking_error_constraint(window.cov(), bench, sign, float(limit))
        elif risk == 'Information Ratio':
            cons += information_ratio_constraint(window, bench, sign, float(limit))
        elif risk == 'Beta':
            cons += beta_constraint(compute_per_asset_betas(window, bench_returns), sign, float(limit))
    try:
        w = RiskAnalysis(window).optimize(key, constraints=cons if cons else False,
                                          bench_weights=bench if key in RELATIVE else None)
        w = np.clip(np.nan_to_num(w), 0, None)
        if w.sum() <= 0:
            raise ValueError("degenerate solution")
        w = w / w.sum()
        # Constraints that conflict can't all hold: the optimizer then returns its best
        # attempt. Say so in the Decisions log (checked on the final weights).
        broken = sum(1 for c in cons
                     if (c['type'] == 'ineq' and np.min(c['fun'](w)) < -1e-4)
                     or (c['type'] == 'eq' and np.max(np.abs(c['fun'](w))) > 1e-4))
        note = f"{broken} constraint(s) not met -- limits may conflict" if broken else ''
        return pd.Series(w, index=cols), note
    except Exception as e:
        return pd.Series(1 / n, index=cols), f"{strategy} failed ({e}); equal weight used"


def regime_backtest(prices, regimes, mapping, asset_constraints=None, risk_constraints=None, custom_weights=None,
                    start=None, end=None, rebalance='Monthly', rebalance_on_change=True, lookback=252,
                    regime_conditional=False, min_obs=60, cost_bps=10.0, te_benchmark='Equal Weight'):
    """Backtest of a strategy whose allocation rule depends on the regime.

    regimes: Series of int regimes (any frequency; forward-filled to the price
             dates, which is causal).
    mapping: {regime: {'Strategy': name, 'Exposure': float}}
    asset_constraints: DataFrame [Regime, Asset, Sign, Limit] -- Regime is an
             int or 'All Regimes'.
    risk_constraints: DataFrame [Regime, Risk, Sign, Limit], Risk in RISK_TYPES.
    custom_weights: DataFrame, index = regime, columns = assets.
    te_benchmark: 'Equal Weight' or a ticker -- what tracking error and
             information ratio are measured against (see benchmark_weights).
    """
    prices = prices.sort_index()
    rets = prices.pct_change(fill_method=None)
    reg_all = regimes.reindex(prices.index, method='ffill')
    idx = prices.loc[pd.Timestamp(start) if start else None:pd.Timestamp(end) if end else None].index
    idx = idx[reg_all.loc[idx].notna().values]
    if len(idx) < 2:
        raise ValueError("No overlap between the backtest dates and the regime history.")
    reg = reg_all.loc[idx].astype(int)

    decisions = {idx[0]} | set(period_end_dates(idx, rebalance)).intersection(idx)
    if rebalance_on_change:
        decisions |= set(idx[reg.ne(reg.shift()).values])
    decisions = sorted(d for d in decisions if d != idx[-1])

    def rows_for(df, regime, columns):
        if df is None or df.empty:
            return []
        sel = df[(df['Regime'] == regime) | (df['Regime'] == 'All Regimes')]
        return sel[columns].values.tolist()

    def decide(date):
        r = int(reg.loc[date])
        spec = mapping.get(r, {'Strategy': 'Equal Weight', 'Exposure': 1.0})
        window = rets.loc[:date].iloc[1:].tail(lookback)
        if regime_conditional:
            same = (reg_all.reindex(window.index) == r).values
            if same.sum() >= min_obs:
                window = window[same]
        live = prices.loc[date].notna()
        window = window.loc[:, live[live].index]
        window = window.loc[:, window.notna().mean() >= 0.9].fillna(0.0)
        custom = custom_weights.loc[r] if custom_weights is not None and r in custom_weights.index else None
        w, note = compute_target_weights(window, spec['Strategy'],
                                         rows_for(asset_constraints, r, ['Asset', 'Sign', 'Limit']),
                                         rows_for(risk_constraints, r, ['Risk', 'Sign', 'Limit']), custom,
                                         bench=benchmark_weights(window.columns, te_benchmark))
        return date, r, w.reindex(prices.columns).fillna(0.0) * float(spec['Exposure']), note

    targets, notes = {}, {}
    decision_regime = {}
    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        for future in as_completed([executor.submit(decide, d) for d in decisions]):
            date, r, w, note = future.result()
            targets[date], decision_regime[date] = w, r
            if note:
                notes[date] = note
    targets = pd.DataFrame(targets).T.sort_index()

    R = rets.loc[idx].fillna(0.0).values
    T, N = R.shape
    target_at = {idx.get_loc(d): targets.loc[d].values for d in targets.index}
    w = np.zeros(N)
    nav, value = np.empty(T), 1.0
    weights, turnover = np.empty((T, N)), np.zeros(T)
    for t in range(T):
        if t > 0:
            port = w @ R[t]
            value *= 1 + port
            if 1 + port > 0:
                w = w * (1 + R[t]) / (1 + port)
        if t in target_at:
            tgt = target_at[t]
            turnover[t] = np.abs(tgt - w).sum()
            value *= 1 - turnover[t] * cost_bps / 1e4
            w = tgt.copy()
        nav[t], weights[t] = value, w

    decisions_df = pd.DataFrame({
        'Regime': pd.Series(decision_regime),
        'Strategy': pd.Series({d: mapping.get(r, {}).get('Strategy', 'Equal Weight') for d, r in decision_regime.items()}),
        'Invested': targets.sum(axis=1),
        'Turnover': pd.Series(turnover, index=idx).reindex(targets.index),
        # reindexed so an empty `notes` doesn't turn the index into plain objects
        'Note': pd.Series(notes, dtype=object).reindex(targets.index),
    }).sort_index()
    return {
        'nav': pd.Series(nav * 100, index=idx),
        'weights': pd.DataFrame(weights, index=idx, columns=prices.columns),
        'targets': targets,
        'turnover': pd.Series(turnover, index=idx),
        'decisions': decisions_df,
        'regimes': reg,
    }


def clean_benchmark_returns(returns):
    """External portfolio returns (dates x portfolios), e.g. the Crypto App's
    `global_returns` or `performance_pct`."""
    if returns is None:
        return pd.DataFrame()
    if isinstance(returns, pd.Series):
        returns = returns.to_frame()
    returns = returns.copy()
    returns.index = pd.to_datetime(returns.index)
    if returns.index.tz is not None:
        returns.index = returns.index.tz_localize(None)
    returns = returns[~returns.index.duplicated(keep='last')].sort_index()
    returns = returns.apply(pd.to_numeric, errors='coerce')
    return returns.dropna(axis=1, how='all')


def benchmark_nav(returns, index):
    """Return series -> level rebased to 100 on the backtest dates.

    Compounded on its own dates first, then read on the backtest dates, so a
    7-day crypto fund lines up with a 5-day ETF backtest (and vice versa)."""
    level = (1 + returns.dropna()).cumprod()
    level = level.reindex(level.index.union(index)).ffill().reindex(index)
    first = level.first_valid_index()
    return level / level.loc[first] * 100 if first is not None else level


def performance_metrics(navs, ann, references=(), turnovers=None):
    """Same metrics as the Crypto App's strategy results (Returns, Volatility,
    Sharpe Ratio, Drawdown, Date Drawdown, Tracking Error to each reference
    series). Each column is measured over its own history (an external
    portfolio may start after the backtest); tracking errors over the overlap."""
    returns = navs.pct_change(fill_method=None)

    def one(name):
        nav = navs[name].dropna()
        rets = nav.pct_change(fill_method=None)
        years = max((nav.index[-1] - nav.index[0]).days / 365.25, 1e-9)
        total = nav.iloc[-1] / nav.iloc[0] - 1
        vol = rets.std() * np.sqrt(ann)
        drawdown = nav / nav.cummax() - 1
        metrics = {
            'Start': nav.index[0].date(),
            'Returns': total,
            'Volatility': vol,
            # annualized return / volatility, as in the Crypto App
            'Sharpe Ratio': ((1 + total) ** (1 / years) - 1) / vol,
            'Drawdown': drawdown.min(),
            'Date Drawdown': drawdown.idxmin().date(),
        }
        for ref in references:
            metrics[f'Tracking Error to {ref}'] = (returns[name] - returns[ref]).std() * np.sqrt(ann)
        if turnovers is not None:
            metrics['Annual Turnover'] = turnovers[name].sum() / years if name in turnovers else np.nan
        return pd.Series(metrics)

    return pd.DataFrame({c: one(c) for c in navs.columns if navs[c].notna().sum() > 1}).fillna(0)


def performance_by_regime(navs, regimes, ann, labels):
    """Annualized return / vol / Sharpe of each series, grouped by the regime
    in force (the regime known at the previous close, when weights were set)."""
    rets = navs.pct_change(fill_method=None)
    in_force = regimes.reindex(rets.index).shift(1)
    grouped = rets.groupby(in_force)
    mean, vol = grouped.mean() * ann, grouped.std() * np.sqrt(ann)
    out = pd.concat({'Ann. Return': mean, 'Ann. Volatility': vol, 'Sharpe': mean / vol}, axis=1)
    out.index = [labels.get(int(k), k) for k in out.index]
    return out.round(4)

## Step 4 - The app

A **scenario bar** above the tabs loads, saves and deletes the settings of the HMM Model, Regime Strategies and Backtest tabs (kept in `HMM_Scenarios.json` with the result of each scenario's last backtest); *Default (initial)* is the built-in scenario the app opens with. Five tabs:

1. **Data**: Yahoo search bar, predefined universe with tick boxes, base currency, start date (as in the Multi Asset App), or a CSV / Excel file; price and cumulative performance charts, return and risk tables.
2. **HMM Model**: feature types (computed on the asset-class baskets, or on an index such as `^GSPC` with its own history), state matching across refits, number of states, covariance type, iterations, restarts, training window, walk-forward refit, state ordering, filtered vs smoothed inference, signal confirmation. Results: asset chart with the regimes shaded, statistics, transition matrices, BIC model selection.
3. **Regime Strategies**: one row per regime (strategy + exposure), per-regime asset constraints, per-regime risk limits (volatility, tracking error, information ratio, beta), editable custom weights grid.
4. **Backtest**: dates, rebalancing (calendar + on regime change), lookback, regime-conditional estimation, costs, benchmarks (static strategy + any loaded asset or external portfolio). Results: performance, drawdown, rolling volatility and rolling correlation (2 × 2, regimes shaded) with the Crypto App strategy metrics underneath, performance by regime, weights through time, decision log.
5. **Compare Scenarios**: metrics, performance and drawdown of saved scenarios over their common period, the settings that differ between them, and a button that refits and backtests the selected scenarios on the prices loaded now.

In [ ]:
# Default scenario the app opens with (edit or clear it in the UI). 5 regimes of ^GSPC
# (return, volatility, drawdown) trained from 2007 on a growing window, refitted every
# quarter with the labels kept across refits: R0 calm bull, R1 normal market,
# R2 correction, R3 bear market, R4 crash. Fully invested, every asset <= 35%. Max
# Information Ratio vs ^GSPC everywhere except the bear regime: tracking error <= 5% and
# beta <= 1 in the bull regimes, <= 10% in corrections and crashes (staying close to the
# index there avoids selling the dips that recover). Bear regime: Minimum Variance with
# T-bills (BIL <= 60%), as bonds can fall with equities (2022), and a commodity sleeve
# (gold >= 10%, broad commodities DBC >= 10%) for inflation-driven bear markets; BIL and
# DBC are only held in that regime.
DEFAULT_SCENARIO = {
    'unchecked': ['DHY'],                       # tickers left out of 'Get Prices'
    'features': INDEX_FEATURES,
    'index': '^GSPC',
    'index_start': datetime.date(2005, 1, 1),   # index history for the features (prices can start later)
    'train_start': datetime.date(2007, 7, 11),  # with 5 restarts an earlier start can stop at a weaker fit
    'train_end': datetime.date(2019, 12, 31),
    'bt_start': datetime.date(2020, 1, 1),
    'n_states': 5,
    'refit': 'Quarterly',
    'window': 'Growing (expanding)',
    'window_years': 5.0,
    'match_states': True,
    'min_prob': 0.6,
    'confirm': 3,
    'strategies': {0: 'Maximum Information Ratio',   # calm bull
                   1: 'Maximum Information Ratio',   # normal market
                   2: 'Maximum Information Ratio',   # correction
                   3: 'Minimum Variance',            # bear market
                   4: 'Maximum Information Ratio'},  # crash / deep drawdown, rebounds included
    'asset_constraints': [{'Regime': r, 'Asset': 'All', 'Sign': '≤', 'Limit': 0.35} for r in range(5)]
                         + [{'Regime': 3, 'Asset': 'BIL', 'Sign': '≤', 'Limit': 0.6},
                            {'Regime': 3, 'Asset': 'GOLD.PA', 'Sign': '≥', 'Limit': 0.1},
                            {'Regime': 3, 'Asset': 'DBC', 'Sign': '≥', 'Limit': 0.1}]
                         + [{'Regime': r, 'Asset': a, 'Sign': '≤', 'Limit': 0.0} for r in (0, 1, 2, 4) for a in ('BIL', 'DBC')],
    'risk_constraints': [{'Regime': r, 'Risk': 'Tracking Error', 'Sign': '≤', 'Limit': 0.05} for r in (0, 1)]
                        + [{'Regime': r, 'Risk': 'Beta', 'Sign': '≤', 'Limit': 1.0} for r in (0, 1)]
                        + [{'Regime': r, 'Risk': 'Tracking Error', 'Sign': '≤', 'Limit': 0.10} for r in (2, 4)],
    'te_benchmark': '^GSPC',
}

# Saved scenarios: the settings of the HMM Model, Regime Strategies and Backtest tabs,
# with the result of their last backtest, in a JSON file next to the notebook.
SCENARIO_FILE = 'HMM_Scenarios.json'
DEFAULT_SCENARIO_NAME = 'Default (initial)'
SCENARIO_DATES = ('index_start', 'train_start', 'train_end', 'bt_start', 'bt_end')


def comparable(value):
    """Settings as plain values (ISO dates, string keys, rounded floats), so two
    copies compare equal after a JSON round trip."""
    if isinstance(value, dict):
        return {str(k): comparable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [comparable(v) for v in value]
    if isinstance(value, float):
        return round(value, 9)
    if isinstance(value, datetime.date):
        return value.isoformat()[:10]
    return value


def same_settings(a, b):
    return comparable(a) == comparable(b)


def settings_from_json(raw):
    """Inverse of comparable() for one scenario's settings: dates and regime numbers back."""
    settings = dict(raw)
    for key in SCENARIO_DATES:
        if settings.get(key):
            settings[key] = datetime.date.fromisoformat(settings[key][:10])
    for key in ('strategies', 'exposures', 'custom_weights'):
        if key in settings:
            settings[key] = {int(r): v for r, v in settings[key].items()}
    for key in ('asset_constraints', 'risk_constraints'):
        settings[key] = [{**c, 'Regime': c['Regime'] if c['Regime'] == 'All Regimes' else int(c['Regime'])}
                         for c in settings.get(key, [])]
    return settings


def load_scenarios(path):
    """{name: {'settings', 'result', 'saved'}} from the scenario file ({} when there is none)."""
    if not os.path.exists(path):
        return {}
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)
    scenarios = {}
    for name, entry in raw.items():
        result = entry.get('result')
        if result:
            result = {**result, 'nav': pd.Series(result['nav']['values'], index=pd.to_datetime(result['nav']['dates']))}
        scenarios[name] = {**entry, 'settings': settings_from_json(entry['settings']), 'result': result}
    return scenarios


def save_scenarios(scenarios, path):
    raw = {}
    for name, entry in scenarios.items():
        result = entry.get('result')
        if result:
            nav = result['nav']
            result = {**result, 'nav': {'dates': [d.strftime('%Y-%m-%d') for d in nav.index],
                                        'values': [round(float(v), 6) for v in nav.values]}}
        raw[name] = {**entry, 'settings': comparable(entry['settings']), 'result': result}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(raw, f, indent=1, ensure_ascii=False)


def display_hmm_app(benchmark_returns=None):
    """benchmark_returns: optional DataFrame of daily returns of other
    portfolios (e.g. the Crypto App's `global_returns`), offered as benchmarks
    in the Backtest tab. Can be replaced later with app['set_benchmarks'](df)."""
    # =========================================================================
    # Shared state / helpers
    # =========================================================================
    state = {
        'prices': pd.DataFrame(),
        'benchmarks': clean_benchmark_returns(benchmark_returns),
        'groups': {},            # asset class -> tickers: the HMM feature baskets
        'result': None,          # output of run_regime_model
        'regimes_daily': None,   # regimes forward-filled on the price dates
        'labels': {},            # regime int -> display label
        'backtest': None,
        'index_prices': None,    # index behind the Index features
        'index_key': None,       # (ticker, start) it was loaded with
        'scenario': None,        # settings of the loaded scenario (dates / mapping / custom weights not shown yet)
        'active_name': DEFAULT_SCENARIO_NAME,   # scenario a new backtest result is kept with
        'fit_settings': None,    # HMM settings the current regimes were fitted with
        'last_run': None,        # settings + NAV of the last backtest
    }
    asset_constraints = [dict(c) for c in DEFAULT_SCENARIO['asset_constraints']]
    risk_constraints = [dict(c) for c in DEFAULT_SCENARIO['risk_constraints']]
    mapping_widgets = {}

    style_wide = {'description_width': '180px'}
    layout_wide = widgets.Layout(width='420px')

    def dark(fig, title, height=450, width=1100, yfmt=None):
        fig.update_layout(title=title, height=height, width=width, plot_bgcolor="black",
                          paper_bgcolor="black", font_color="white",
                          font=dict(family="Arial Narrow", size=13), legend_title_text='')
        if yfmt:
            fig.update_layout(yaxis_tickformat=yfmt)
        return fig

    def regime_colors(n):
        if n == 1:
            return ['rgb(26,152,80)']
        return px.colors.sample_colorscale('RdYlGn_r', [i / (n - 1) for i in range(n)])

    def add_regime_shading(fig, regimes, opacity=0.22):
        """One rectangle per contiguous regime episode + a legend swatch per regime."""
        regimes = regimes.dropna().astype(int)
        if regimes.empty:
            return fig
        n = len(state['labels'])
        colors = regime_colors(n)
        starts = regimes.index[regimes.ne(regimes.shift()).values]
        ends = list(starts[1:]) + [regimes.index[-1]]
        shapes = [dict(type='rect', xref='x', yref='paper', x0=s, x1=e, y0=0, y1=1,
                       fillcolor=colors[regimes.loc[s] % n], opacity=opacity, line_width=0, layer='below')
                  for s, e in zip(starts, ends)]
        fig.update_layout(shapes=list(fig.layout.shapes) + shapes)
        for k, label in state['labels'].items():
            fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=label,
                                     marker=dict(size=12, symbol='square', color=colors[k])))
        return fig

    def show(output, *items):
        with output:
            output.clear_output(wait=True)
            for item in items:
                display(item)

    def table(df, digits=4):
        return display_scrollable_df(df.round(digits) if hasattr(df, 'round') else df)

    def set_options(dd, options, fallback=None):
        """Reassign Dropdown options while keeping the current value (see Crypto App).
        Options may be plain values or (label, value) pairs."""
        options = list(options)
        values = [o[1] if isinstance(o, tuple) else o for o in options]
        old = dd.value
        dd.options = options
        if old in values:
            dd.value = old
        elif values:
            dd.value = fallback if fallback in values else values[0]

    # =========================================================================
    # TAB 1 -- DATA (same universe / search / currency inputs as the Multi Asset App)
    # =========================================================================
    universe_table = pd.DataFrame([(t, asset_class) for asset_class, lst in UNIVERSE.items() for t in lst],
                                  columns=['Ticker', 'Asset Class']).set_index('Ticker')
    universe_state = {
        'table': universe_table[~universe_table.index.duplicated()],
        'tickers': [],
        'unchecked': set(DEFAULT_SCENARIO['unchecked']),   # tickers the user untick -- kept across refreshes
        'show_details': False,
        'source': None,           # 'yahoo' or 'file' -- what the loaded prices came from
    }

    start_date = widgets.DatePicker(
        value=datetime.date(2015, 1, 1), description='Starting Date of Prices',
        style={'description_width': '200px'}, layout=widgets.Layout(width='350px'))
    loading_bar = widgets.IntProgress(description='Loading prices...', min=0, max=100, style={'description_width': '150px'})
    data_button = widgets.Button(description='Get Prices', button_style='info')
    base_currency_dd = widgets.Dropdown(
        description='Base Currency', options=list(dict.fromkeys((BASE_CURRENCY,) + tuple(BASE_CURRENCY_CHOICES))),
        value=BASE_CURRENCY, style={'description_width': '120px'}, layout=widgets.Layout(width='230px'))
    convert_checkbox = widgets.Checkbox(value=CONVERT_TO_BASE, description='Convert prices to base currency',
                                        style={'description_width': 'initial'})
    scope_output = widgets.Output()
    main_output = widgets.Output()
    price_output = widgets.Output()

    def scope_update(_=None):
        """Redraw the universe table and its checkboxes (tick = include in 'Get Prices')."""
        selected = universe_state['table']
        selected_list = list(selected.index)
        universe_view = selected
        if universe_state['show_details']:
            universe_view = selected.join(fetch_ticker_details(selected_list))
        with scope_output:
            scope_output.clear_output(wait=True)
            display(display_scrollable_df(universe_view))
            checkboxes = {t: widgets.Checkbox(description=t, value=t not in universe_state['unchecked'])
                          for t in selected_list}
            rows = [widgets.HBox(list(checkboxes.values())[i:i + 5]) for i in range(0, len(checkboxes), 5)]

            def on_change(change=None):
                for k, cb in checkboxes.items():
                    (universe_state['unchecked'].discard if cb.value else universe_state['unchecked'].add)(k)
                universe_state['tickers'] = [k for k, cb in checkboxes.items() if cb.value]

            for cb in checkboxes.values():
                cb.observe(on_change, names="value")
            on_change()
            display(Markdown("### Selected Tickers"))
            display(widgets.VBox(rows))

    scope_update()

    # --- search Yahoo Finance and add tickers to the universe ---
    search_box = widgets.Text(continuous_update=False,   # value only changes on Enter / leaving the box
                              placeholder='Name or ticker, e.g. "Nvidia", "gold ETF", "EXV5.DE"',
                              description='Search', layout=widgets.Layout(width='450px'))
    search_button = widgets.Button(description='Search', button_style='info', icon='search')
    search_results = widgets.Dropdown(description='Result', options=[], layout=widgets.Layout(width='650px'))
    asset_class_choice = widgets.Combobox(
        description='Asset Class', placeholder='Auto (from Yahoo type)',
        options=sorted(set(universe_state['table']['Asset Class']) | set(QUOTE_TYPE_CLASS.values())),
        ensure_option=False)
    add_ticker_button = widgets.Button(description='Add to Universe', button_style='success', icon='plus')
    details_button = widgets.Button(description='Load Details', icon='info-circle',
                                    tooltip='Name, exchange, currency, volume, market cap... from yfinance')
    search_output = widgets.Output()
    search_hits = {}

    def on_search(_=None):
        search_hits.clear()
        with search_output:
            search_output.clear_output()
            hits = search_yahoo(search_box.value)
            if hits.empty:
                search_results.options = []
                print(f"No match for '{search_box.value}'.")
                return
            for r in hits.itertuples(index=False):
                label = f"{r.Symbol} — {r.Name} ({r.Type}, {r.Exchange})"
                search_hits[label] = r._asdict()
            set_options(search_results, list(search_hits))

    def on_add_ticker(_=None):
        with search_output:
            search_output.clear_output()
            hit = search_hits.get(search_results.value)
            symbol = hit['Symbol'] if hit else search_box.value.strip().upper()
            if not symbol:
                print("Search for a ticker first.")
                return
            if symbol in universe_state['table'].index:
                print(f"{symbol} is already in the universe.")
                return
            # only add what yfinance can actually price
            if not has_price_history(symbol):
                print(f"❌ yfinance returns no prices for {symbol} -- not added.")
                return
            quote_type = (hit or {}).get('Type') or fetch_ticker_details([symbol]).iloc[0].get('Type') or ''
            asset_class = asset_class_choice.value.strip() or QUOTE_TYPE_CLASS.get(str(quote_type).upper(), 'Other')
            universe_state['table'] = pd.concat(
                [universe_state['table'], pd.DataFrame({'Asset Class': [asset_class]}, index=pd.Index([symbol], name='Ticker'))])
            asset_class_choice.options = sorted(set(asset_class_choice.options) | {asset_class})
            print(f"✅ Added {symbol} ({asset_class}). Click 'Get Prices' to load it.")
        scope_update()

    def on_details(_=None):
        universe_state['show_details'] = True
        with search_output:
            search_output.clear_output()
            print("Loading details from yfinance...")
        scope_update()
        with search_output:
            search_output.clear_output()

    search_box.observe(lambda ch: on_search() if ch['new'].strip() else None, names='value')
    search_button.on_click(on_search)
    add_ticker_button.on_click(on_add_ticker)
    details_button.on_click(on_details)
    search_ui = widgets.VBox([
        widgets.HBox([search_box, search_button]),
        widgets.HBox([search_results]),
        widgets.HBox([asset_class_choice, add_ticker_button, details_button]),
        search_output])

    # --- optional: prices from a CSV / Excel file instead of Yahoo ---
    file_path = widgets.Text(value='MA_Price.xlsx', description='File Path', layout=widgets.Layout(width='450px'))
    file_upload = widgets.FileUpload(accept='.csv,.xlsx,.xls', multiple=False, description='Upload')
    file_button = widgets.Button(description='Load File', button_style='info')

    def render_universe_charts():
        prices = state['prices']
        shown = [t for t in ('^GSPC', 'SPY') if t in prices.columns] or list(prices.columns[:1])
        with price_output:
            price_output.clear_output(wait=True)
            price_graph = widgets.Output()
            return_graph = widgets.Output()
            with price_graph:
                fig = px.line(prices, title="Price", width=800, height=400, render_mode='svg')
                fig.update_traces(visible="legendonly", selector=lambda t: t.name not in shown)
                dark(fig, "Price", height=400, width=800).show()
            with return_graph:
                cumulative_returns = prices.pct_change(fill_method=None).fillna(0)
                cumulative_returns = (1 + cumulative_returns).cumprod() * 100
                fig2 = px.line(cumulative_returns, render_mode='svg')
                fig2.update_traces(visible="legendonly", selector=lambda t: t.name not in shown)
                dark(fig2, "Cumulative Performance", height=400, width=800).show()
            display(widgets.HBox([price_graph, return_graph]))
            display(display_scrollable_df(prices))

    def after_load(prices, messages, source):
        state['prices'] = prices
        state['result'] = None
        state['backtest'] = None
        universe_state['source'] = source
        with main_output:
            main_output.clear_output(wait=True)
            for message in messages:
                print(message)
            print(f"✅ Loaded prices for {prices.shape[1]} assets from {prices.index[0].date()} to "
                  f"{prices.index[-1].date()} ({len(prices)} dates, {periods_per_year(prices.index)} periods / year)")
            late = [c for c in prices.columns if prices[c].first_valid_index() > prices.index[0]]
            if late:
                print(f"ℹ️ Shorter history (enters the backtest once it has data): {', '.join(late)}")
            common = prices.dropna()
            if len(common) > 2:
                display(display_scrollable_df(get_asset_returns(common)))
                display(display_scrollable_df(get_asset_risk(common)))
        on_prices_loaded()
        render_universe_charts()

    def get_prices(_=None):
        with main_output:
            main_output.clear_output(wait=True)
            tickers = universe_state['tickers']
            if not tickers:
                print("No tickers selected.")
                return
            start = start_date.value
            if not isinstance(start, datetime.date) or start >= datetime.date.today():
                print("Please select a valid start date (in the past).")
                return
            loading_bar.value = 0
            loading_bar.max = len(tickers)
            display(loading_bar)

            def progress(_ticker, _ok):
                loading_bar.value += 1

            try:
                prices, messages = load_yahoo_prices(tickers, start, base_currency=base_currency_dd.value,
                                                     convert=convert_checkbox.value, on_progress=progress)
            except Exception as e:
                print("❌ Error while loading prices:", e)
                return
            if prices.empty:
                print("❌ No prices could be loaded.")
                return
        after_load(prices, messages, 'yahoo')

    def get_file_prices(_=None):
        with main_output:
            main_output.clear_output(wait=True)
            try:
                prices = clean_prices(read_price_file(file_path.value, file_upload.value))
                if prices.empty:
                    raise ValueError("no numeric price columns found")
            except Exception as e:
                print("❌ Error while reading the file:", e)
                return
        name = file_upload.value[0]['name'] if file_upload.value else file_path.value
        after_load(prices, [f"📄 Prices from {name} (no currency conversion)"], 'file')

    def on_currency_settings(_=None):
        # same as the Multi Asset App: changing the currency reloads the Yahoo prices
        if universe_state['source'] == 'yahoo':
            get_prices()

    data_button.on_click(get_prices)
    file_button.on_click(get_file_prices)
    start_date.observe(lambda ch: get_prices() if ch['new'] else None, names='value')
    base_currency_dd.observe(on_currency_settings, names='value')
    convert_checkbox.observe(on_currency_settings, names='value')

    data_ui = widgets.VBox([
        search_ui,
        widgets.HBox([base_currency_dd, convert_checkbox]),
        widgets.HBox([start_date, data_button]),
        scope_output,
        widgets.HTML("<b>Or load prices from a file</b> (CSV / Excel: dates in the first column, one column per asset)"),
        widgets.HBox([file_path, file_upload, file_button]),
        main_output, price_output,
    ])

    # =========================================================================
    # TAB 2 -- HMM MODEL
    # =========================================================================
    feature_groups_html = widgets.HTML("<i>Load prices to see the asset-class baskets.</i>")
    feature_checks = {f: widgets.Checkbox(value=f in DEFAULT_SCENARIO['features'], description=f, indent=False,
                                          layout=widgets.Layout(width='200px'))
                      for f in FEATURE_TYPES}
    frequency = widgets.Dropdown(description='Observation Frequency', options=['Daily', 'Weekly'], style=style_wide, layout=layout_wide)
    smooth_window = widgets.IntText(value=5, description='Return Smoothing (obs)', style=style_wide, layout=layout_wide)
    vol_window = widgets.IntText(value=21, description='Vol / Corr Window (obs)', style=style_wide, layout=layout_wide)
    index_ticker = widgets.Text(value=DEFAULT_SCENARIO['index'], description='Index (Index features)', style=style_wide, layout=layout_wide)
    index_start = widgets.DatePicker(value=DEFAULT_SCENARIO['index_start'], description='Index History Start',
                                     style=style_wide, layout=layout_wide)

    n_states = widgets.IntSlider(value=DEFAULT_SCENARIO['n_states'], min=2, max=6, description='Number of States', style=style_wide, layout=layout_wide)
    cov_type = widgets.Dropdown(description='Covariance Type', options=COVARIANCE_TYPES, style=style_wide, layout=layout_wide)
    n_iter = widgets.IntText(value=500, description='Max Iterations', style=style_wide, layout=layout_wide)
    tol = widgets.FloatText(value=1e-4, description='Tolerance', style=style_wide, layout=layout_wide)
    n_init = widgets.IntSlider(value=5, min=1, max=20, description='Random Restarts', style=style_wide, layout=layout_wide)
    seed = widgets.IntText(value=42, description='Random Seed', style=style_wide, layout=layout_wide)

    train_start = widgets.DatePicker(description='Training Start', style=style_wide, layout=layout_wide)
    train_end = widgets.DatePicker(description='Training End', style=style_wide, layout=layout_wide)
    refit = widgets.Dropdown(description='Walk-forward Refit', options=list(REFIT_OPTIONS), value=DEFAULT_SCENARIO['refit'], style=style_wide, layout=layout_wide)
    window = widgets.Dropdown(description='Training Window', options=WINDOW_OPTIONS, value=DEFAULT_SCENARIO['window'], style=style_wide, layout=layout_wide)
    window_years = widgets.BoundedFloatText(value=DEFAULT_SCENARIO['window_years'], min=0.5, max=50.0, step=0.5, description='Sliding Length (years)',
                                            disabled=DEFAULT_SCENARIO['window'] != 'Sliding', style=style_wide, layout=layout_wide)
    inference = widgets.Dropdown(description='Inference', options=INFERENCE_OPTIONS, style=style_wide, layout=layout_wide)
    order_by = widgets.Dropdown(description='Order States By', options=ORDER_OPTIONS, style=style_wide, layout=layout_wide)
    match_states = widgets.Checkbox(value=DEFAULT_SCENARIO['match_states'], indent=False,
                                    description='Keep labels across refits (match states to the first fit)')
    min_prob = widgets.FloatSlider(value=DEFAULT_SCENARIO['min_prob'], min=0.0, max=0.95, step=0.05, description='Min Prob to Switch', style=style_wide, layout=layout_wide)
    confirm = widgets.IntText(value=DEFAULT_SCENARIO['confirm'], description='Confirmation (obs)', style=style_wide, layout=layout_wide)
    chart_asset = widgets.Dropdown(description='Chart Asset', style=style_wide, layout=layout_wide)

    fit_btn = widgets.Button(description='Fit HMM', button_style='primary')
    select_btn = widgets.Button(description='Model Selection (BIC)')
    hmm_status = widgets.Output()
    regime_chart_out = widgets.Output()
    stats_out = widgets.Output()
    transition_out = widgets.Output()
    selection_out = widgets.Output()

    def index_prices():
        """Prices of the index behind the Index features, loaded once per (ticker, start)."""
        key = (index_ticker.value.strip(), index_start.value)
        if state['index_key'] != key:
            data = clean_prices(get_close([key[0]], start=key[1], end=datetime.date.today()))
            if data.empty:
                raise ValueError(f"No prices for the index {key[0]}.")
            state['index_prices'], state['index_key'] = data.iloc[:, 0], key
        return state['index_prices']

    def feature_settings():
        feature_types = [f for f, cb in feature_checks.items() if cb.value]
        return dict(groups=state['groups'], feature_types=feature_types,
                    vol_window=max(2, vol_window.value), smooth_window=max(1, smooth_window.value),
                    frequency=frequency.value,
                    index_prices=index_prices() if set(feature_types) & set(INDEX_FEATURES) else None)

    def check_ready(output):
        with output:
            output.clear_output(wait=True)
            if state['prices'].empty:
                print("⚠️ Load prices first (Data tab).")
                return False
            if not train_start.value or not train_end.value or train_start.value >= train_end.value:
                print("⚠️ Invalid training window.")
                return False
        return True

    def make_labels(result):
        """'R0 · vol 12% · ret +9%' from the feature basket stats in each regime (training window)."""
        ref = state['ref_returns']
        ann = 52 if frequency.value == 'Weekly' else periods_per_year(ref.index)
        in_sample = result['regimes'].loc[:result['train_end']]
        r = ref.reindex(in_sample.index)
        labels = {}
        for k in range(n_states.value):
            sub = r[in_sample == k]
            if len(sub) > 1:
                labels[k] = f"R{k} · vol {sub.std() * np.sqrt(ann):.0%} · ret {sub.mean() * ann:+.0%}"
            else:
                labels[k] = f"R{k}"
        return labels

    def on_fit(_):
        if not check_ready(hmm_status):
            return
        with hmm_status:
            print("Fitting HMM...")
            try:
                settings = feature_settings()
                features = build_features(state['prices'], fit_end=pd.Timestamp(train_end.value), **settings)
                # with the Index features, the index orders and describes the states
                state['ref_returns'] = (index_returns(settings['index_prices'], settings['frequency'])
                                        if settings['index_prices'] is not None
                                        else reference_returns(state['prices'], settings['groups'], settings['frequency']))
                result = run_regime_model(
                    features, state['ref_returns'], train_start.value, train_end.value,
                    n_states=n_states.value, covariance_type=cov_type.value, n_iter=n_iter.value, tol=tol.value,
                    n_init=n_init.value, random_state=seed.value, refit=refit.value, order_by=order_by.value,
                    inference=inference.value, min_prob=min_prob.value, confirm=max(1, confirm.value),
                    window=window.value, window_years=window_years.value, match_states=match_states.value)
            except Exception as e:
                hmm_status.clear_output(wait=True)
                print("❌ HMM error:", e)
                return
            state['result'] = result
            state['fit_settings'] = hmm_settings()
            state['labels'] = make_labels(result)
            state['regimes_daily'] = result['regimes'].reindex(state['prices'].index, method='ffill')
            state['backtest'] = None
            hmm_status.clear_output(wait=True)
            current = result['regimes'].iloc[-1]
            last_probs = result['probs'].iloc[-1]
            print(f"✅ {len(result['fits'])} fit(s), {result['features'].shape[1]} features, "
                  f"{len(result['features'])} observations from {result['features'].index[0].date()}.")
            print(f"Current regime ({result['probs'].index[-1].date()}): {state['labels'][current]}  "
                  f"| probabilities: " + ', '.join(f"R{k} {p:.0%}" for k, p in last_probs.items()))
            if inference.value.startswith('Smoothed'):
                print("⚠️ Smoothed probabilities use future data: do not trust a backtest built on them.")
        render_regimes()
        rebuild_strategy_tab()

    def render_regimes(_=None):
        result = state['result']
        if result is None or state['prices'].empty:
            return
        labels = state['labels']
        regimes = result['regimes']
        asset = chart_asset.value or state['prices'].columns[0]

        series = state['prices'][asset].loc[regimes.index[0]:].dropna()
        fig = go.Figure(go.Scatter(x=series.index, y=series.values, name=asset, line=dict(color='white', width=1.4)))
        add_regime_shading(fig, state['regimes_daily'].loc[series.index[0]:])
        fig.add_vline(x=result['train_end'], line_dash='dash', line_color='deepskyblue')
        fig.add_annotation(x=result['train_end'], y=1.02, yref='paper', text='end of training',
                           showarrow=False, font=dict(color='deepskyblue'))
        dark(fig, f'{asset} with HMM regimes')
        fig.update_yaxes(type='log')
        show(regime_chart_out, fig)

        ann = periods_per_year(state['prices'].index)
        mean, vol, sharpe = regime_asset_statistics(state['prices'], state['regimes_daily'], ann)
        rename = lambda df: df.rename(index=lambda k: labels.get(int(k), k))
        feats = result['features']
        feature_means = feats.groupby(regimes.reindex(feats.index)).mean().rename(index=lambda k: labels[int(k)])
        show(stats_out,
             Markdown('### Regime Summary'), table(regime_summary(regimes, result['transmat'], labels)),
             Markdown('### Average Features per Regime'), table(feature_means),
             Markdown('### Annualized Return per Regime'), table(rename(mean)),
             Markdown('### Annualized Volatility per Regime'), table(rename(vol)),
             Markdown('### Sharpe Ratio per Regime'), table(rename(sharpe), 2))

        named = lambda df: df.rename(index=labels, columns=labels)
        fits = pd.DataFrame([{k: f[k] for k in ('Fit date', 'Window start', 'Observations', 'Log-Likelihood')}
                             for f in result['fits']])
        fits['Fit date'] = fits['Fit date'].dt.date
        fits['Window start'] = fits['Window start'].dt.date
        heat = px.imshow(named(result['transmat']), text_auto='.2f', color_continuous_scale='Blues', aspect='auto')
        dark(heat, 'Transition matrix (last fitted model)', height=400, width=700)
        show(transition_out,
             heat,
             Markdown('### Empirical Transitions (signal after confirmation)'),
             table(named(empirical_transitions(regimes))),
             Markdown('### Fits'), table(fits))

    def on_select(_):
        if not check_ready(selection_out):
            return
        with selection_out:
            print("Fitting 2 to 6 states...")
            try:
                features = build_features(state['prices'], fit_end=pd.Timestamp(train_end.value), **feature_settings())
                scores = select_n_states(features, train_start.value, train_end.value, max_states=6,
                                         covariance_type=cov_type.value, n_iter=n_iter.value, tol=tol.value,
                                         n_init=min(n_init.value, 3), random_state=seed.value)
            except Exception as e:
                selection_out.clear_output(wait=True)
                print("❌ Model selection error:", e)
                return
            selection_out.clear_output(wait=True)
            fig = px.line(scores[['AIC', 'BIC']], markers=True)
            dark(fig, 'Information criteria (lower is better)', height=350, width=700)
            display(table(scores, 1))
            display(fig)
            print(f"Lowest BIC: {scores['BIC'].idxmin()} states.")

    fit_btn.on_click(on_fit)
    select_btn.on_click(on_select)
    chart_asset.observe(render_regimes, names='value')

    def on_window(change=None):
        window_years.disabled = window.value != 'Sliding'
    window.observe(on_window, names='value')

    feature_grid = widgets.GridBox(list(feature_checks.values()),
                                   layout=widgets.Layout(grid_template_columns='repeat(3, 210px)'))
    hmm_settings_ui = widgets.VBox([
        widgets.HTML("<h3>Features</h3>"),
        feature_groups_html,
        widgets.HBox([frequency, smooth_window, vol_window]),
        feature_grid,
        widgets.HBox([index_ticker, index_start]),
        widgets.HTML("<h3>Model</h3>"),
        widgets.HBox([widgets.VBox([n_states, cov_type, n_iter]), widgets.VBox([tol, n_init, seed])]),
        widgets.HTML("<h3>Training & Inference</h3>"),
        widgets.HBox([widgets.VBox([train_start, train_end, refit]), widgets.VBox([window, window_years, inference, order_by, match_states])]),
        widgets.HTML("<h3>Signal</h3>"),
        widgets.HBox([min_prob, confirm]),
        widgets.HBox([fit_btn, select_btn]),
        hmm_status, selection_out,
    ])
    hmm_tabs = widgets.Tab(children=[
        hmm_settings_ui,
        widgets.VBox([chart_asset, regime_chart_out]),
        stats_out,
        transition_out,
    ])
    for i, title in enumerate(['Settings', 'Regimes', 'Statistics', 'Transitions']):
        hmm_tabs.set_title(i, title)

    # =========================================================================
    # TAB 3 -- REGIME STRATEGIES (strategy, exposure and constraints per regime)
    # =========================================================================
    mapping_box = widgets.VBox()
    mapping_out = widgets.Output()

    cons_regime = widgets.Dropdown(description='Regime', options=['All Regimes'], style={'description_width': '80px'})
    cons_asset = widgets.Dropdown(description='Asset', options=['All'], style={'description_width': '60px'})
    cons_sign = widgets.Dropdown(description='Sign', options=['=', '≥', '≤'], value='≤', style={'description_width': '50px'}, layout=widgets.Layout(width='140px'))
    cons_limit = widgets.FloatText(value=0.3, description='Limit', style={'description_width': '50px'}, layout=widgets.Layout(width='160px'))
    add_cons_btn = widgets.Button(description='Add Constraint', button_style='success')
    clear_cons_btn = widgets.Button(description='Clear All', button_style='danger')
    cons_out = widgets.Output()

    risk_regime = widgets.Dropdown(description='Regime', options=['All Regimes'], style={'description_width': '80px'})
    risk_type = widgets.Dropdown(description='Risk Type', options=RISK_TYPES, style={'description_width': '80px'})
    te_benchmark = widgets.Dropdown(description='TE / IR / Beta Benchmark', options=['Equal Weight', DEFAULT_SCENARIO['te_benchmark']],
                                    value=DEFAULT_SCENARIO['te_benchmark'],
                                    style={'description_width': '180px'}, layout=widgets.Layout(width='400px'))
    risk_sign = widgets.Dropdown(description='Sign', options=['≤', '≥', '='], style={'description_width': '50px'}, layout=widgets.Layout(width='140px'))
    risk_limit = widgets.FloatText(value=0.15, description='Limit', style={'description_width': '50px'}, layout=widgets.Layout(width='160px'))
    add_risk_btn = widgets.Button(description='Add Risk Limit', button_style='success')
    clear_risk_btn = widgets.Button(description='Clear All', button_style='danger')
    risk_out = widgets.Output()

    custom_grid = DataGrid(pd.DataFrame(), editable=True, layout={"height": "220px"})
    equal_fill_btn = widgets.Button(description='Fill Equal Weight')
    zero_fill_btn = widgets.Button(description='Reset to 0')
    custom_status = widgets.Output()

    def regime_from_label(label):
        if label == 'All Regimes':
            return label
        return {v: k for k, v in state['labels'].items()}[label]

    def default_mapping(k, n):
        """(strategy, exposure) of regime k: the loaded scenario's when it has n regimes."""
        scenario = state['scenario']
        if n == scenario['n_states'] and k in scenario['strategies']:
            return scenario['strategies'][k], scenario['exposures'].get(k, 1.0)
        # calm -> chase returns, middle -> balance risk, stressed -> defend
        if k == 0:
            return 'Sharpe Ratio', 1.0
        return ('Minimum Variance' if k == n - 1 else 'Risk Parity'), 1.0

    def rebuild_strategy_tab():
        n = len(state['labels'])
        rows = [widgets.HBox([widgets.HTML('<b>Regime</b>', layout=widgets.Layout(width='300px')),
                              widgets.HTML('<b>Strategy</b>', layout=widgets.Layout(width='260px')),
                              widgets.HTML('<b>Exposure</b>', layout=widgets.Layout(width='120px'))])]
        # edits are kept on a refit with the same number of regimes (with another, R2 means something else)
        old = ({k: (w['Strategy'].value, w['Exposure'].value) for k, w in mapping_widgets.items()}
               if len(mapping_widgets) == n else {})
        mapping_widgets.clear()
        for k, label in state['labels'].items():
            strategy_value, exposure_value = old.get(k, default_mapping(k, n))
            mapping_widgets[k] = {
                'Strategy': widgets.Dropdown(options=list(STRATEGIES), value=strategy_value, layout=widgets.Layout(width='250px')),
                'Exposure': widgets.BoundedFloatText(value=exposure_value, min=0, max=1, step=0.05, layout=widgets.Layout(width='100px')),
            }
            swatch = f"<span style='color:{regime_colors(n)[k]}'>■</span> {label}"
            rows.append(widgets.HBox([widgets.HTML(swatch, layout=widgets.Layout(width='300px')),
                                      mapping_widgets[k]['Strategy'], mapping_widgets[k]['Exposure']]))
        mapping_box.children = rows

        regime_options = ['All Regimes'] + list(state['labels'].values())
        set_options(cons_regime, regime_options)
        set_options(risk_regime, regime_options)
        # Constraints reference regime numbers; with a different number of
        # states they may point to regimes that no longer exist.
        for store in (asset_constraints, risk_constraints):
            store[:] = [c for c in store if c['Regime'] == 'All Regimes' or c['Regime'] in state['labels']]
        render_constraints()
        rebuild_custom_grid()
        show(mapping_out, Markdown(f"✅ {n} regimes. Strategies are applied to the assets loaded in the Data tab."))

    def rebuild_custom_grid():
        if state['prices'].empty or not state['labels']:
            return
        index = [state['labels'][k] for k in state['labels']]
        old = custom_grid.data if custom_grid.data is not None else pd.DataFrame()
        grid = pd.DataFrame(0.0, index=index, columns=state['prices'].columns)
        if not old.empty:
            grid.update(old.reindex(index=grid.index, columns=grid.columns))
        else:   # new grid: the loaded scenario's weights
            for k, row in state['scenario'].get('custom_weights', {}).items():
                if k in state['labels']:
                    for asset, weight in row.items():
                        if asset in grid.columns:
                            grid.loc[state['labels'][k], asset] = weight
        custom_grid.data = grid

    def fill_custom(equal):
        if custom_grid.data is None or custom_grid.data.empty:
            return
        data = pd.DataFrame(custom_grid.data)
        custom_grid.data = pd.DataFrame(1 / data.shape[1] if equal else 0.0, index=data.index, columns=data.columns)

    def constraints_frame(store, columns):
        if not store:
            return pd.DataFrame(columns=columns)
        df = pd.DataFrame(store)
        df['Regime'] = df['Regime'].map(lambda r: state['labels'].get(r, r))
        return df[columns]

    def render_constraints():
        show(cons_out, table(constraints_frame(asset_constraints, ['Regime', 'Asset', 'Sign', 'Limit'])))
        show(risk_out, table(constraints_frame(risk_constraints, ['Regime', 'Risk', 'Sign', 'Limit'])))

    def on_add_constraint(_):
        asset_constraints.append({'Regime': regime_from_label(cons_regime.value), 'Asset': cons_asset.value,
                                  'Sign': cons_sign.value, 'Limit': cons_limit.value})
        render_constraints()

    def on_add_risk(_):
        risk_constraints.append({'Regime': regime_from_label(risk_regime.value), 'Risk': risk_type.value,
                                 'Sign': risk_sign.value, 'Limit': risk_limit.value})
        render_constraints()

    def on_clear(store):
        store.clear()
        render_constraints()

    add_cons_btn.on_click(on_add_constraint)
    clear_cons_btn.on_click(lambda _: on_clear(asset_constraints))
    add_risk_btn.on_click(on_add_risk)
    clear_risk_btn.on_click(lambda _: on_clear(risk_constraints))
    equal_fill_btn.on_click(lambda _: fill_custom(True))
    zero_fill_btn.on_click(lambda _: fill_custom(False))
    render_constraints()

    strategy_ui = widgets.VBox([
        widgets.HTML("<h3>Strategy per Regime</h3>"
                     "<p>Exposure &lt; 1 keeps the rest in cash (0% return). "
                     "'Custom Weights' uses the grid at the bottom.</p>"),
        mapping_box, mapping_out,
        widgets.HTML("<h3>Asset Constraints per Regime</h3>"
                     "<p>Asset 'All' applies the limit to every asset (e.g. All ≤ 0.3 caps each weight at 30%); "
                     "an asset's own row with the same sign replaces it for that asset (All ≤ 0.35 + BIL ≤ 0.6 "
                     "caps BIL at 60%). Only used by the optimized strategies, except a cap at 0 (≤ 0 or = 0): "
                     "that asset is left out of the regime whatever the strategy, Equal Weight included "
                     "(e.g. BIL ≤ 0 keeps T-bills out of a regime).</p>"),
        widgets.HBox([cons_regime, cons_asset, cons_sign, cons_limit]),
        widgets.HBox([add_cons_btn, clear_cons_btn]), cons_out,
        widgets.HTML("<h3>Risk Limits per Regime</h3><p>Annualized and ex-ante, on the fully-invested weights "
                     "(before the exposure scaling); only used by the optimized strategies. "
                     "Volatility and Tracking Error in decimals (0.05 = 5%), Information Ratio and Beta as ratios "
                     "(0.5, 0.8). Tracking error, information ratio, beta and the <b>Minimum Tracking Error</b> / "
                     "<b>Maximum Information Ratio</b> strategies are measured against the TE / IR / Beta benchmark: "
                     "equal weight of the assets available at each date, or 100% in one asset (e.g. ^GSPC). Beta is "
                     "estimated on the lookback window of each decision.</p>"),
        te_benchmark,
        widgets.HBox([risk_regime, risk_type, risk_sign, risk_limit]),
        widgets.HBox([add_risk_btn, clear_risk_btn]), risk_out,
        widgets.HTML("<h3>Custom Weights per Regime</h3>"),
        widgets.HBox([equal_fill_btn, zero_fill_btn]), custom_grid, custom_status,
    ])

    # =========================================================================
    # TAB 4 -- BACKTEST
    # =========================================================================
    bt_start = widgets.DatePicker(description='Backtest Start', style=style_wide, layout=layout_wide)
    bt_end = widgets.DatePicker(description='Backtest End', style=style_wide, layout=layout_wide)
    rebalance = widgets.Dropdown(description='Rebalancing', options=list(REBALANCING), value='Monthly', style=style_wide, layout=layout_wide)
    on_change = widgets.Checkbox(value=True, description='Also rebalance on regime change', indent=False)
    lookback = widgets.IntText(value=252, description='Lookback (days)', style=style_wide, layout=layout_wide)
    regime_conditional = widgets.Checkbox(value=False, description='Estimate on same-regime days only', indent=False)
    min_obs = widgets.IntText(value=60, description='Min Same-Regime Obs', style=style_wide, layout=layout_wide)
    cost_bps = widgets.FloatText(value=10.0, description='Transaction Cost (bps)', style=style_wide, layout=layout_wide)
    bench_strategy = widgets.Dropdown(description='Static Benchmark', options=[s for s in STRATEGIES if s not in ('Custom Weights', 'Cash')],
                                      value='Equal Weight', style=style_wide, layout=layout_wide)
    bench_asset = widgets.Dropdown(description='Benchmark', options=[('None', 'none')], style=style_wide, layout=layout_wide)

    def refresh_bench_options(fallback=None):
        # values are 'asset:<ticker>' / 'fund:<portfolio>' so a portfolio can
        # share a name with a ticker
        options = [('None', 'none')]
        options += [(a, f'asset:{a}') for a in state['prices'].columns]
        options += [(f'{c} (portfolio)', f'fund:{c}') for c in state['benchmarks'].columns]
        set_options(bench_asset, options, fallback=fallback)

    refresh_bench_options()
    window_vol = widgets.IntText(value=252, description='Vol Window:', style=style_wide, layout=layout_wide)
    window_corr = widgets.IntText(value=252, description='Rolling Correlation:', style=style_wide, layout=layout_wide)
    run_btn = widgets.Button(description='Run Backtest', button_style='primary')

    bt_status = widgets.Output()
    perf_out = widgets.Output()
    regime_perf_out = widgets.Output()
    weights_out = widgets.Output()
    decisions_out = widgets.Output()

    def current_mapping():
        return {k: {'Strategy': w['Strategy'].value, 'Exposure': w['Exposure'].value} for k, w in mapping_widgets.items()}

    def custom_weights_frame():
        if custom_grid.data is None or custom_grid.data.empty:
            return None
        data = pd.DataFrame(custom_grid.data).apply(pd.to_numeric, errors='coerce').fillna(0.0)
        data.index = [regime_from_label(label) for label in data.index]
        return data

    def on_run(_):
        with bt_status:
            bt_status.clear_output(wait=True)
            if state['result'] is None:
                print("⚠️ Fit the HMM first (HMM Model tab).")
                return
            if not same_settings(hmm_settings(), state['fit_settings']):
                print("⚠️ The HMM settings changed since the last fit (e.g. a scenario was loaded): click Fit HMM first.")
                return
            if not bt_start.value or not bt_end.value or bt_start.value >= bt_end.value:
                print("⚠️ Invalid backtest dates.")
                return
            mapping = current_mapping()
            custom = custom_weights_frame()
            if custom is not None:
                for k, spec in mapping.items():
                    total = custom.loc[k].sum() if k in custom.index else 0
                    if spec['Strategy'] == 'Custom Weights' and not 0 < total <= 1 + 1e-6:
                        print(f"⚠️ Custom weights of {state['labels'][k]} sum to {total:.2f} (must be in (0, 1]).")
                        return
            if bt_start.value <= train_end.value:
                print("⚠️ The backtest starts inside the HMM training window: those regimes are in-sample.")
            print("Running backtest...")
            prices = state['prices']
            common = dict(start=bt_start.value, end=bt_end.value, rebalance=rebalance.value,
                          lookback=max(20, lookback.value), cost_bps=cost_bps.value,
                          te_benchmark=te_benchmark.value)
            try:
                bt = regime_backtest(
                    prices, state['regimes_daily'], mapping,
                    asset_constraints=pd.DataFrame(asset_constraints) if asset_constraints else None,
                    risk_constraints=pd.DataFrame(risk_constraints) if risk_constraints else None,
                    custom_weights=custom, rebalance_on_change=on_change.value,
                    regime_conditional=regime_conditional.value, min_obs=min_obs.value, **common)
                # Static benchmark: one strategy whatever the regime, same dates / costs, no constraints
                constant = pd.Series(0, index=state['regimes_daily'].dropna().index)
                bench = regime_backtest(prices, constant, {0: {'Strategy': bench_strategy.value, 'Exposure': 1.0}},
                                        rebalance_on_change=False, **common)
            except Exception as e:
                bt_status.clear_output(wait=True)
                print("❌ Backtest error:", e)
                return

            navs = pd.DataFrame({'Regime Strategy': bt['nav'], f'Static {bench_strategy.value}': bench['nav']}).dropna()
            kind, _, name = (bench_asset.value or 'none').partition(':')
            if kind == 'asset' and name in prices.columns:
                navs[name] = benchmark_nav(prices[name].pct_change(fill_method=None), navs.index)
            elif kind == 'fund' and name in state['benchmarks'].columns:
                navs[name] = benchmark_nav(state['benchmarks'][name], navs.index)
                if navs[name].isna().all():
                    print(f"⚠️ Portfolio '{name}' has no returns over the backtest dates.")
            # the chosen benchmark if any, else the static strategy: reference of the correlation chart
            reference = name if name in navs.columns else navs.columns[1]
            state['backtest'] = {'bt': bt, 'navs': navs, 'reference': reference}
            state['last_run'] = {'settings': current_scenario(), 'nav': bt['nav'],
                                 'run': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
                                 'data': f"{prices.shape[1]} assets, {prices.index[0].date()} to {prices.index[-1].date()}"}
            kept = keep_result()
            bt_status.clear_output(wait=True)
            print(f"✅ {len(bt['decisions'])} rebalancing decisions from {navs.index[0].date()} to {navs.index[-1].date()}.")
            if kept:
                print(f"💾 Result kept with scenario '{state['active_name']}' (Compare Scenarios tab).")
            notes = bt['decisions']['Note'].dropna()
            unmet = notes.str.contains('not met').sum()
            if unmet:
                print(f"⚠️ {unmet} decisions could not meet every constraint -- the limits may conflict "
                      f"(e.g. a tight TE vs one asset with an asset cap). See the Note column in Decisions.")
            if len(notes) - unmet:
                print(f"⚠️ {len(notes) - unmet} optimizations failed and fell back to equal weight (see Decisions).")
        render_backtest()

    def line_chart(frame, title, yfmt=None):
        """800 x 400 line chart with the regime in force shaded, in its own Output (for 2 x 2 grids)."""
        fig = go.Figure([go.Scatter(x=frame.index, y=frame[c], name=c) for c in frame.columns])
        add_regime_shading(fig, state['backtest']['bt']['regimes'])
        dark(fig, title, height=400, width=800, yfmt=yfmt)
        out = widgets.Output()
        with out:
            fig.show()
        return out

    def render_backtest(_=None):
        if not state['backtest']:
            return
        navs, bt = state['backtest']['navs'], state['backtest']['bt']
        reference = state['backtest']['reference']
        labels = state['labels']
        ann = periods_per_year(state['prices'].index)
        regimes = bt['regimes']

        returns = navs.pct_change(fill_method=None)
        drawdown = navs / navs.cummax() - 1
        rolling_vol = returns.rolling(max(2, window_vol.value)).std() * np.sqrt(ann)
        rolling_corr = returns.drop(columns=reference).rolling(max(2, window_corr.value)).corr(returns[reference])

        static = navs.columns[1]
        references = list(dict.fromkeys([reference, static]))
        metrics = performance_metrics(navs, ann, references, turnovers={'Regime Strategy': bt['turnover']})
        show(perf_out,
             widgets.HBox([line_chart(navs, 'Performance'), line_chart(drawdown, 'Drawdown', '.2%')]),
             widgets.HBox([line_chart(rolling_vol, 'Portfolio Rolling Volatility', '.2%'),
                           line_chart(rolling_corr, f'Rolling Correlation to {reference}')]),
             Markdown('### Returns'), table(rebalanced_metrics(navs.dropna())),
             Markdown('### Strategy Metrics'), table(metrics.astype(object)))

        by_regime = performance_by_regime(navs, regimes, ann, labels)
        time_in = regimes.shift(1).value_counts(normalize=True).rename(index=lambda k: labels[int(k)])
        fig_bar = px.bar(by_regime['Sharpe'].reset_index().melt(id_vars='index'), x='index', y='value',
                         color='variable', barmode='group', labels={'index': '', 'value': 'Sharpe'})
        dark(fig_bar, 'Sharpe ratio by regime in force', height=380, width=900)
        show(regime_perf_out,
             Markdown('### Performance by Regime in Force'), table(by_regime),
             Markdown('### Time Spent in Each Regime (backtest)'), table(time_in.to_frame('% of Time')),
             fig_bar)

        weights = bt['weights'].loc[:, (bt['weights'].abs() > 1e-4).any()]
        weekly = weights.resample('W-FRI').last()
        weekly['Cash'] = (1 - weekly.sum(axis=1)).clip(lower=0)
        fig_w = go.Figure([go.Scatter(x=weekly.index, y=weekly[c], name=c, mode='lines')
                           for c in weekly.columns])
        add_regime_shading(fig_w, regimes)
        dark(fig_w, 'Portfolio weights (end of week, after drift)', yfmt='.0%')
        targets = bt['targets'].loc[:, (bt['targets'].abs() > 1e-4).any()]
        show(weights_out, fig_w, Markdown('### Target Weights at Each Decision'), table(targets))

        decisions = bt['decisions'].copy()
        decisions['Regime'] = decisions['Regime'].map(labels)
        decisions.index = decisions.index.date
        show(decisions_out, table(decisions))

    run_btn.on_click(on_run)
    window_vol.observe(render_backtest, names='value')
    window_corr.observe(render_backtest, names='value')

    backtest_settings = widgets.VBox([
        widgets.HBox([widgets.VBox([bt_start, bt_end, rebalance, on_change]),
                      widgets.VBox([lookback, regime_conditional, min_obs, cost_bps])]),
        widgets.HBox([bench_strategy, bench_asset]),
        widgets.HBox([window_vol, window_corr]),
        run_btn, bt_status,
    ])
    backtest_tabs = widgets.Tab(children=[perf_out, regime_perf_out, weights_out, decisions_out])
    for i, title in enumerate(['Performance', 'By Regime', 'Weights', 'Decisions']):
        backtest_tabs.set_title(i, title)
    backtest_ui = widgets.VBox([backtest_settings, backtest_tabs])

    # =========================================================================
    # SCENARIOS -- save / load the settings of the HMM Model, Regime Strategies and
    # Backtest tabs; each keeps the result of its last backtest for the Compare tab
    # =========================================================================
    setting_widgets = {
        'frequency': frequency, 'smooth_window': smooth_window, 'vol_window': vol_window,
        'index': index_ticker, 'index_start': index_start,
        'n_states': n_states, 'cov_type': cov_type, 'n_iter': n_iter, 'tol': tol, 'n_init': n_init, 'seed': seed,
        'train_start': train_start, 'train_end': train_end, 'refit': refit, 'window': window,
        'window_years': window_years, 'inference': inference, 'order_by': order_by,
        'match_states': match_states, 'min_prob': min_prob, 'confirm': confirm,
        'te_benchmark': te_benchmark,
        'bt_start': bt_start, 'bt_end': bt_end, 'rebalance': rebalance, 'rebalance_on_change': on_change,
        'lookback': lookback, 'regime_conditional': regime_conditional, 'min_obs': min_obs,
        'cost_bps': cost_bps, 'bench_strategy': bench_strategy,
    }
    # what the regimes depend on: changing one of them needs a new fit
    hmm_keys = ['frequency', 'smooth_window', 'vol_window', 'index', 'index_start', 'n_states', 'cov_type', 'n_iter',
                'tol', 'n_init', 'seed', 'train_start', 'train_end', 'refit', 'window', 'window_years', 'inference',
                'order_by', 'match_states', 'min_prob', 'confirm']

    def hmm_settings():
        settings = {k: setting_widgets[k].value for k in hmm_keys}
        settings['features'] = [f for f, cb in feature_checks.items() if cb.value]
        return settings

    def current_scenario():
        """Settings of the HMM Model, Regime Strategies and Backtest tabs."""
        scenario = {k: w.value for k, w in setting_widgets.items()}
        scenario['features'] = [f for f, cb in feature_checks.items() if cb.value]
        for key in ('index_start', 'train_start', 'train_end', 'bt_start'):
            if scenario[key] is None:                 # prices not loaded yet: the loaded scenario's dates
                scenario[key] = state['scenario'].get(key)
        n = scenario['n_states']
        fitted = len(mapping_widgets) == n            # the strategy rows match the number of states
        if fitted:
            pairs = {k: (w['Strategy'].value, w['Exposure'].value) for k, w in mapping_widgets.items()}
        else:   # rows from a fit with another number of states (or none): the scenario's / generic mapping
            pairs = {k: default_mapping(k, n) for k in range(n)}
        scenario['strategies'] = {k: s for k, (s, _) in pairs.items()}
        scenario['exposures'] = {k: e for k, (_, e) in pairs.items()}
        # rows for regimes the model doesn't have are dropped at the fit (see rebuild_strategy_tab)
        in_model = lambda r: r == 'All Regimes' or r < n
        scenario['asset_constraints'] = [dict(c) for c in asset_constraints if in_model(c['Regime'])]
        scenario['risk_constraints'] = [dict(c) for c in risk_constraints if in_model(c['Regime'])]
        custom = custom_weights_frame() if fitted else None
        if custom is None:
            weights = state['scenario'].get('custom_weights', {}) if state['scenario']['n_states'] == n else {}
        else:
            weights = {int(k): {a: float(v) for a, v in row.items() if v} for k, row in custom.iterrows() if row.any()}
        scenario['custom_weights'] = {k: dict(v) for k, v in weights.items() if k < n}
        if not state['prices'].empty and scenario['bt_end'] == state['prices'].index[-1].date():
            scenario['bt_end'] = None                 # 'to the last price', whatever the data
        return scenario

    def full_scenario(settings):
        """A saved scenario completed with the default of every setting it doesn't have."""
        scenario = {**copy.deepcopy(state['default_scenario']), **copy.deepcopy(settings)}
        scenario['features'] = [f for f in FEATURE_TYPES if f in scenario['features']]
        return scenario

    default = {k: w.value for k, w in setting_widgets.items()}            # the widgets start on DEFAULT_SCENARIO
    default.update({k: copy.deepcopy(v) for k, v in DEFAULT_SCENARIO.items() if k != 'unchecked'})
    default.update(features=list(DEFAULT_SCENARIO['features']), bt_end=None, custom_weights={},
                   exposures={k: 1.0 for k in DEFAULT_SCENARIO['strategies']})
    state['default_scenario'] = default
    state['scenario'] = copy.deepcopy(default)

    scenario_out = widgets.Output()
    try:
        saved = load_scenarios(SCENARIO_FILE)
        state['scenario_file_error'] = ''
    except Exception as e:   # unreadable file: start without it and never overwrite it
        saved = {}
        state['scenario_file_error'] = (f"⚠️ Could not read {SCENARIO_FILE} ({e}): saving is disabled "
                                        "until the file is fixed or removed.")
    stored_default = saved.pop(DEFAULT_SCENARIO_NAME, None)
    state['scenarios'] = {DEFAULT_SCENARIO_NAME: {
        'settings': copy.deepcopy(default), 'saved': 'built-in',
        # a stored result is kept only while the built-in default is unchanged
        'result': stored_default['result'] if stored_default and same_settings(stored_default['settings'], default) else None}}
    state['scenarios'].update({name: {**entry, 'settings': full_scenario(entry['settings'])} for name, entry in saved.items()})

    def persist():
        """Write the scenario file; returns an error message ('' when fine)."""
        if state['scenario_file_error']:
            return state['scenario_file_error']
        try:
            save_scenarios(state['scenarios'], SCENARIO_FILE)
        except Exception as e:
            return f"⚠️ Could not write {SCENARIO_FILE}: {e}"
        return ''

    scenario_dd = widgets.Dropdown(description='Scenario', style={'description_width': '70px'}, layout=widgets.Layout(width='380px'))
    load_btn = widgets.Button(description='Load', icon='folder-open',
                              tooltip='Put the scenario settings in the HMM Model, Regime Strategies and Backtest tabs')
    scenario_name = widgets.Text(placeholder='Name', description='Save as', style={'description_width': '60px'},
                                 layout=widgets.Layout(width='320px'))
    save_btn = widgets.Button(description='Save', button_style='success', icon='save',
                              tooltip='Save the current settings (with their backtest result once it has run)')
    delete_btn = widgets.Button(description='Delete', button_style='danger', icon='trash')
    compare_select = widgets.SelectMultiple(description='Scenarios', rows=8, style={'description_width': '80px'},
                                            layout=widgets.Layout(width='650px'))
    compare_btn = widgets.Button(description='Compare', button_style='primary', icon='bar-chart')
    run_selected_btn = widgets.Button(description='Run selected on current prices', button_style='warning', icon='play',
                                      layout=widgets.Layout(width='280px'))
    compare_status = widgets.Output()
    compare_out = widgets.Output()

    def refresh_scenario_lists(selected=None):
        names = list(state['scenarios'])
        set_options(scenario_dd, names, fallback=selected)
        if selected in names:
            scenario_dd.value = selected
        chosen = [n for n in compare_select.value if n in names]
        compare_select.options = [(f"{n}  ({e['result']['run'] if e.get('result') else 'not run'})", n)
                                  for n, e in state['scenarios'].items()]
        compare_select.value = tuple(chosen)

    def keep_result():
        """Attach the last backtest to the active scenario when its settings are the ones that ran."""
        entry, run = state['scenarios'].get(state['active_name']), state['last_run']
        if entry is None or run is None or not same_settings(entry['settings'], run['settings']):
            return False
        entry['result'] = {k: run[k] for k in ('nav', 'run', 'data')}
        error = persist()
        refresh_scenario_lists(selected=state['active_name'])
        if error:
            with scenario_out:
                print(error)
        return True

    def apply_scenario(name):
        """Load a scenario's settings in the widgets; returns the settings that couldn't be restored."""
        scenario = full_scenario(state['scenarios'][name]['settings'])
        state['scenario'], state['active_name'] = scenario, name
        set_options(te_benchmark, list(dict.fromkeys(list(te_benchmark.options) + [scenario['te_benchmark']])))
        skipped = []
        for key, w in setting_widgets.items():
            value = scenario.get(key)
            if key == 'bt_end' and value is None and not state['prices'].empty:
                value = state['prices'].index[-1].date()
            if value is None:
                continue
            try:
                w.value = value
            except Exception:
                skipped.append(key)
        for f, cb in feature_checks.items():
            cb.value = f in scenario['features']
        on_window()
        asset_constraints[:] = [dict(c) for c in scenario['asset_constraints']]
        risk_constraints[:] = [dict(c) for c in scenario['risk_constraints']]
        mapping_widgets.clear()
        custom_grid.data = pd.DataFrame()
        if state['labels'] and len(state['labels']) == scenario['n_states']:
            rebuild_strategy_tab()
        else:   # the strategy rows appear with the next fit
            mapping_box.children = []
            render_constraints()
            show(mapping_out, Markdown(f"Fit the HMM ({scenario['n_states']} states) to see the strategy of each regime."))
        return skipped

    def on_load(_):
        name = scenario_dd.value
        if not name:
            return
        skipped = apply_scenario(name)
        scenario_name.value = '' if name == DEFAULT_SCENARIO_NAME else name
        with scenario_out:
            scenario_out.clear_output()
            print(f"📂 '{name}' loaded: click Fit HMM, then Run Backtest (the Data tab is unchanged).")
            if skipped:
                print(f"⚠️ Not restored (value no longer valid): {', '.join(skipped)}")

    def on_save(_):
        name = scenario_name.value.strip()
        with scenario_out:
            scenario_out.clear_output()
            if not name or name == DEFAULT_SCENARIO_NAME:
                print("⚠️ Type a name for the scenario (the built-in default can't be overwritten).")
                return
            settings = current_scenario()
            run, previous = state['last_run'], state['scenarios'].get(name)
            if run is not None and same_settings(run['settings'], settings):
                result = {k: run[k] for k in ('nav', 'run', 'data')}
            elif previous and previous.get('result') and same_settings(previous['settings'], settings):
                result = previous['result']
            else:
                result = None
            state['scenarios'][name] = {'settings': settings, 'result': result,
                                        'saved': datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
            state['scenario'], state['active_name'] = full_scenario(settings), name
            error = persist()
            refresh_scenario_lists(selected=name)
            print(f"💾 '{name}' {'updated' if previous else 'saved'}" +
                  (" with its backtest result." if result else
                   " (no result yet: Run Backtest and it is kept with this scenario)."))
            if len(mapping_widgets) != settings['n_states']:
                print(f"ℹ️ No fit with {settings['n_states']} states yet: the strategies saved are the scenario's or "
                      "the generic ones. Fit the HMM, adjust them in Regime Strategies and save again if needed.")
            if error:
                print(error)

    def on_delete(_):
        name = scenario_dd.value
        with scenario_out:
            scenario_out.clear_output()
            if name == DEFAULT_SCENARIO_NAME:
                print("⚠️ The built-in default can't be deleted.")
                return
            state['scenarios'].pop(name, None)
            if state['active_name'] == name:
                state['active_name'] = None
            error = persist()
            refresh_scenario_lists(selected=DEFAULT_SCENARIO_NAME)
            print(f"🗑️ '{name}' deleted." + (f" {error}" if error else ''))

    def render_compare(_=None):
        names = list(compare_select.value)
        done = {n: state['scenarios'][n]['result'] for n in names if state['scenarios'][n].get('result')}
        with compare_status:
            compare_status.clear_output()
            missing = [n for n in names if n not in done]
            if missing:
                print(f"Not run yet (use 'Run selected on current prices'): {', '.join(missing)}")
            if not done:
                print("Select scenarios that have a result.")
        if not done:
            compare_out.clear_output()
            return
        common = pd.DataFrame({n: r['nav'] for n, r in done.items()}).dropna()
        if len(common) < 2:
            show(compare_out, Markdown("⚠️ The selected backtests have no period in common."))
            return
        kind, _, asset = (bench_asset.value or 'none').partition(':')
        if kind == 'asset' and asset in state['prices'].columns and asset not in common.columns:
            common[asset] = benchmark_nav(state['prices'][asset].pct_change(fill_method=None), common.index)
        common = common / common.iloc[0] * 100
        metrics = performance_metrics(common, periods_per_year(common.index))
        runs = pd.DataFrame({n: {'Backtest': f"{r['nav'].index[0].date()} to {r['nav'].index[-1].date()}",
                                 'Data': r['data'], 'Run': r['run']} for n, r in done.items()})
        # settings that differ between the compared scenarios
        settings = {n: comparable(state['scenarios'][n]['settings']) for n in done}
        keys = sorted({k for s in settings.values() for k in s})
        differ = {k: {n: settings[n].get(k) for n in settings} for k in keys
                  if len({json.dumps(settings[n].get(k), sort_keys=True) for n in settings}) > 1}
        differ = pd.DataFrame(differ).T.map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)
        fig = go.Figure([go.Scatter(x=common.index, y=common[c], name=c) for c in common.columns])
        dark(fig, f'Performance over the common period (rebased to 100)')
        drawdown = common / common.cummax() - 1
        fig_dd = go.Figure([go.Scatter(x=drawdown.index, y=drawdown[c], name=c) for c in drawdown.columns])
        dark(fig_dd, 'Drawdown', height=350, yfmt='.0%')
        show(compare_out,
             Markdown(f"### Metrics from {common.index[0].date()} to {common.index[-1].date()} (common period)"),
             table(metrics.astype(object)), fig, fig_dd,
             Markdown('### Settings that differ'),
             table(differ) if len(differ) else Markdown('The compared scenarios have the same settings.'),
             Markdown('### Runs'), table(runs))

    def on_run_selected(_):
        names = list(compare_select.value)
        with compare_status:
            compare_status.clear_output()
            if state['prices'].empty:
                print("⚠️ Load prices first (Data tab).")
                return
            if not names:
                print("Select the scenarios to run.")
                return
        for i, name in enumerate(names, 1):
            with compare_status:
                print(f"[{i}/{len(names)}] {name}: fitting the HMM and running the backtest...")
            before = state['last_run']
            apply_scenario(name)
            on_fit(None)
            on_run(None)
            ok = state['last_run'] is not before and state['scenarios'][name].get('result') is not None \
                and same_settings(state['last_run']['settings'], state['scenarios'][name]['settings'])
            with compare_status:
                print(f"    {'✅ done' if ok else '❌ failed: see the HMM Model / Backtest tabs'}")
        refresh_scenario_lists(selected=state['active_name'])
        render_compare()

    load_btn.on_click(on_load)
    save_btn.on_click(on_save)
    delete_btn.on_click(on_delete)
    compare_btn.on_click(render_compare)
    run_selected_btn.on_click(on_run_selected)
    refresh_scenario_lists(selected=DEFAULT_SCENARIO_NAME)
    with scenario_out:
        print(state['scenario_file_error'] or
              f"'{DEFAULT_SCENARIO_NAME}' loaded; {len(state['scenarios']) - 1} saved scenario(s) in {SCENARIO_FILE}.")

    scenario_bar = widgets.VBox([widgets.HBox([scenario_dd, load_btn, scenario_name, save_btn, delete_btn]), scenario_out])
    compare_ui = widgets.VBox([
        widgets.HTML("<h3>Compare Scenarios</h3><p>A scenario keeps the result of its last backtest (Run Backtest "
                     "after loading or saving it). Results are compared over the period they have in common. "
                     "<b>Run selected on current prices</b> refits and backtests the selected scenarios on the "
                     "prices loaded now, so they are compared on the same data.</p>"),
        compare_select, widgets.HBox([compare_btn, run_selected_btn]), compare_status, compare_out,
    ])

    # =========================================================================
    # Wiring: new prices -> refresh every asset / date selector
    # =========================================================================
    def on_prices_loaded():
        prices = state['prices']
        assets = list(prices.columns)
        defaults = [a for a in ('^GSPC', 'SPY', 'BTC-USD') if a in assets] or assets[:1]
        state['groups'] = {g: t for g, t in asset_groups(assets, universe_state['table']).items()
                           if g not in NO_FEATURE_BASKET}
        feature_groups_html.value = (
            "<b>Feature baskets</b> (equal weight per asset class, from the universe table):<br>" +
            "<br>".join(f"&nbsp;&nbsp;<b>{g}</b> ({len(t)}): {', '.join(t)}" for g, t in state['groups'].items()))
        set_options(chart_asset, assets, fallback=defaults[0])
        refresh_bench_options(fallback=f'asset:{defaults[0]}')
        if bench_asset.value == 'none':
            bench_asset.value = f'asset:{defaults[0]}'
        set_options(cons_asset, assets + ['All'])
        set_options(te_benchmark, ['Equal Weight'] + assets)

        first, last = prices.index[0], prices.index[-1]
        # the loaded scenario's dates when the prices cover them, else split the history in two.
        # Training may start before the prices: the Index features have their own history.
        scenario = state['scenario']
        scenario_end = pd.Timestamp(scenario['train_end']) if scenario.get('train_end') else None
        if scenario_end is not None and first < scenario_end < last:
            train_start.value = scenario.get('train_start') or first.date()
            train_end.value = scenario_end.date()
            start = pd.Timestamp(scenario['bt_start']) if scenario.get('bt_start') else scenario_end
            bt_start.value = max(start, scenario_end + pd.Timedelta(days=1)).date()
        else:
            train_start.value = first.date()
            train_end.value = (first + (last - first) * 0.5).date()
            bt_start.value = (first + (last - first) * 0.5 + pd.Timedelta(days=1)).date()
        end = scenario.get('bt_end')
        bt_end.value = end if end and first < pd.Timestamp(end) <= last else last.date()

        for out in (hmm_status, regime_chart_out, stats_out, transition_out,
                    bt_status, perf_out, regime_perf_out, weights_out, decisions_out):
            out.clear_output()
        render_constraints()     # the constraints belong to the scenario: kept when the prices change
        rebuild_custom_grid()

    # =========================================================================
    # TAB ASSEMBLY
    # =========================================================================
    main_tabs = widgets.Tab(children=[data_ui, hmm_tabs, strategy_ui, backtest_ui, compare_ui])
    for i, title in enumerate(['Data', 'HMM Model', 'Regime Strategies', 'Backtest', 'Compare Scenarios']):
        main_tabs.set_title(i, title)

    def set_benchmarks(returns):
        """Replace the external portfolios offered as benchmarks (e.g. after re-running the Crypto App)."""
        state['benchmarks'] = clean_benchmark_returns(returns)
        refresh_bench_options()
        print(f"✅ {state['benchmarks'].shape[1]} portfolios available as benchmarks.")

    display(widgets.VBox([scenario_bar, main_tabs]))
    # handles for scripting / debugging from the notebook
    return {'state': state, 'tabs': main_tabs, 'scenario_bar': scenario_bar, 'set_benchmarks': set_benchmarks}

## Step 5 - Launch

**Other portfolios as benchmarks.** `benchmark_returns` takes any DataFrame of daily returns (dates × portfolios). Each column is added to the *Benchmark* dropdown of the Backtest tab as `<name> (portfolio)`. If the Crypto App ran in the same kernel, its globals are picked up automatically:

* `global_returns`: Fund, Core, Overlay, Bitcoin and every `Buy and Hold` / `Rebalanced` allocation of the grid (set by *Refresh* in the Strategy Return tab),
* `performance_pct`: Fund, Core, Overlay and Bitcoin (set by the rolling optimization).

After re-running the Crypto App, refresh the list without restarting this app: `app['set_benchmarks'](global_returns)`.

In [ ]:
benchmark_returns = None
for name in ('global_returns', 'performance_pct'):
    candidate = globals().get(name)
    if isinstance(candidate, pd.DataFrame) and not candidate.empty:
        benchmark_returns = candidate
        print(f"Using the Crypto App's `{name}` as benchmark portfolios: {', '.join(map(str, candidate.columns))}")
        break

app = display_hmm_app(benchmark_returns=benchmark_returns)